In [21]:
# Cell 1: Setup and List Contents
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Set the path to the unzipped data directory
data_path = r'C:\Users\riddh\Downloads\DA-E-Commerce-Data-Challenge-main'
print(f'Data directory: {data_path}')
print(f'Directory exists: {os.path.exists(data_path)}')
print(f'\nContents:')
for item in os.listdir(data_path):
    print(f'  - {item}')


Data directory: C:\Users\riddh\Downloads\DA-E-Commerce-Data-Challenge-main
Directory exists: True

Contents:
  - buyer.csv
  - LICENSE
  - metadata.pdf
  - sales.csv
  - Vendor Datasets


In [22]:

buyers = pd.read_csv(os.path.join(data_path, 'buyer.csv'))
print(f'Buyer data shape: {buyers.shape}')
print(f'\nBuyer data columns: {list(buyers.columns)}')
print(f'\nFirst few rows:')
print(buyers.head())

Buyer data shape: (52500, 14)

Buyer data columns: ['buyer_id', 'customer_group', 'is_active_buyer', 'signup_date', 'customer_segment', 'preferred_subcategory', 'subcategory_pool', 'preferred_channel', 'preferred_payment', 'region', 'state', 'timezone', 'is_referred', 'wishlist_size']

First few rows:
  buyer_id customer_group  is_active_buyer    signup_date customer_segment  \
0   B12962       Frequent                1  12/10/20 0:00   Bargain Hunter   
1   B27489        Regular                1   5/17/23 0:00     Beauty Lover   
2   B41901        Dormant                0   7/27/23 0:00  Tech Enthusiast   
3   B50973        Dormant                0   6/26/18 0:00  Fashion Shopper   
4   B13680       Frequent                1    7/6/21 0:00   Cozy Homemaker   

  preferred_subcategory                 subcategory_pool preferred_channel  \
0                 Socks   Small Accessories, Phone Cases            Mobile   
1            Foundation              Lipsticks, Skincare               A

In [23]:
print("=" * 80)
print("BUYER DATASET EDA")
print("=" * 80)

# 1. Basic info
print(f"\nShape: {buyers.shape}")
print(f"Columns: {list(buyers.columns)}")

# 2. Nulls & duplicates
print(f"\nNull values:\n{buyers.isnull().sum()}")
print(f"Duplicate buyer_ids: {buyers.duplicated(subset=['buyer_id']).sum()}")

# 3. Parse signup_date
buyers['signup_date'] = pd.to_datetime(buyers['signup_date'], format='%m/%d/%y %H:%M', errors='coerce')

# 4. Customer groups
print(f"\nCustomer Group:\n{buyers['customer_group'].value_counts()}")

# 5. Active status
print(f"\nActive Buyers:\n{buyers['is_active_buyer'].value_counts()}")

# 6. Wishlist stats
print(f"\nWishlist Size:\n{buyers['wishlist_size'].describe()}")

# 7. Top segments
print(f"\nTop 10 Segments:\n{buyers['customer_segment'].value_counts().head(10)}")

# 8. Geographic
print(f"\nRegion Distribution:\n{buyers['region'].value_counts()}")

# 9. Referrals
print(f"\nReferred Buyers:\n{buyers['is_referred'].value_counts()}")

# 10. Summary
print("\n" + "=" * 80)
print(f"Total Buyers: {len(buyers):,}")
print(f"Active: {(buyers['is_active_buyer'] == 1).sum():,} ({(buyers['is_active_buyer'] == 1).sum() / len(buyers) * 100:.1f}%)")
print(f"Referred: {(buyers['is_referred'] == 1).sum():,} ({(buyers['is_referred'] == 1).sum() / len(buyers) * 100:.1f}%)")
print(f"Avg Wishlist: {buyers['wishlist_size'].mean():.1f}")
print(f"Unique Segments: {buyers['customer_segment'].nunique()}")
print("=" * 80)


BUYER DATASET EDA

Shape: (52500, 14)
Columns: ['buyer_id', 'customer_group', 'is_active_buyer', 'signup_date', 'customer_segment', 'preferred_subcategory', 'subcategory_pool', 'preferred_channel', 'preferred_payment', 'region', 'state', 'timezone', 'is_referred', 'wishlist_size']

Null values:
buyer_id                 2634
customer_group              0
is_active_buyer             0
signup_date               519
customer_segment            0
preferred_subcategory       0
subcategory_pool            0
preferred_channel           0
preferred_payment           0
region                   5239
state                       0
timezone                 2624
is_referred                 0
wishlist_size               0
dtype: int64
Duplicate buyer_ids: 4999

Customer Group:
customer_group
Dormant       26270
Regular       10488
Occasional     7852
Frequent       6311
VIP            1579
Name: count, dtype: int64

Active Buyers:
is_active_buyer
0    26270
1    26230
Name: count, dtype: int64

Wishli

In [24]:
# CHECK IF is_active_buyer IS REDUNDANT

print("=" * 80)
print("CHECKING: Is is_active_buyer REDUNDANT?")
print("=" * 80)

# 1. CHECK: Can it be derived from customer_group?
print("\n1. RELATIONSHIP WITH CUSTOMER_GROUP")
print("-" * 80)
cross = pd.crosstab(buyers['customer_group'], buyers['is_active_buyer'], margins=True)
print(cross)
print("\nQuestion: Is 'Dormant' always 0? Or do some Dormant=1 exist?")

dormant_active = buyers[(buyers['customer_group'] == 'Dormant') & (buyers['is_active_buyer'] == 1)]
print(f"Dormant buyers who are active: {len(dormant_active)}")
if len(dormant_active) > 0:
    print("✅ NOT REDUNDANT - customer_group ≠ is_active_buyer")
else:
    print("⚠️ POSSIBLY REDUNDANT - Dormant always inactive")

# 2. CHECK: Can it be derived from sales data?
print("\n2. COMPARE WITH SALES DATA")
print("-" * 80)

# Load sales data
sales = pd.read_csv(os.path.join(data_path, 'sales.csv'))
active_in_sales = sales['buyer_id'].unique()

print(f"Buyers in buyer.csv: {buyers['buyer_id'].nunique():,}")
print(f"Buyers in sales.csv: {len(active_in_sales):,}")

# Check if is_active_buyer matches presence in sales
buyers['in_sales'] = buyers['buyer_id'].isin(active_in_sales).astype(int)

# Compare
compare = pd.crosstab(buyers['is_active_buyer'], buyers['in_sales'], margins=True)
print(f"\nComparison:")
print(compare)

# Check if they match
matches = (buyers['is_active_buyer'] == buyers['in_sales']).sum()
total = len(buyers)
match_pct = matches / total * 100

print(f"\nMatches: {matches:,} / {total:,} ({match_pct:.1f}%)")
print(f"Mismatches: {total - matches:,}")

if match_pct > 95:
    print("⚠️ REDUNDANT - is_active_buyer matches sales presence")
else:
    print("✅ NOT REDUNDANT - has independent information")

# 3. SHOW MISMATCHES
print("\n3. MISMATCHES")
print("-" * 80)

mismatch = buyers[buyers['is_active_buyer'] != buyers['in_sales']]
print(f"Total mismatches: {len(mismatch)}")

if len(mismatch) > 0:
    print(f"\nExamples:")
    print(f"  is_active_buyer=1 but NOT in sales:")
    inactive_but_marked = mismatch[mismatch['is_active_buyer'] == 1]
    print(f"    Count: {len(inactive_but_marked)}")
    
    print(f"\n  is_active_buyer=0 but IN sales:")
    active_but_marked = mismatch[mismatch['is_active_buyer'] == 0]
    print(f"    Count: {len(active_but_marked)}")
    print(f"    Sample orders per buyer: {active_but_marked['buyer_id'].map(lambda x: (sales['buyer_id'] == x).sum()).describe()}")

# 4. VARIANCE CHECK
print("\n4. PREDICTIVE VALUE")
print("-" * 80)

# Does it vary with other features?
features_to_check = ['customer_group', 'region', 'preferred_channel', 'is_referred']
variance = {}

for feat in features_to_check:
    # For each value of feature, what's the active rate?
    active_rate = buyers.groupby(feat)['is_active_buyer'].mean()
    variance[feat] = active_rate.std()  # Higher std = more variation = more predictive
    print(f"{feat:20s}: Active% by category = {active_rate.min():.1%} to {active_rate.max():.1%}")

# 5. VERDICT
print("\n" + "=" * 80)
print("VERDICT")
print("=" * 80)

if match_pct > 98:
    print("❌ REDUNDANT - is_active_buyer = presence in sales")
    print("   Recommendation: DROP this column")
elif len(dormant_active) == 0:
    print("⚠️ SEMI-REDUNDANT - Almost same as customer_group='Dormant'")
    print("   Recommendation: KEEP for clarity, but maybe not critical")
else:
    print("✅ NOT REDUNDANT - Has independent meaning")
    print("   Recommendation: KEEP this column")

print("=" * 80)


CHECKING: Is is_active_buyer REDUNDANT?

1. RELATIONSHIP WITH CUSTOMER_GROUP
--------------------------------------------------------------------------------
is_active_buyer      0      1    All
customer_group                      
Dormant          26270      0  26270
Frequent             0   6311   6311
Occasional           0   7852   7852
Regular              0  10488  10488
VIP                  0   1579   1579
All              26270  26230  52500

Question: Is 'Dormant' always 0? Or do some Dormant=1 exist?
Dormant buyers who are active: 0
⚠️ POSSIBLY REDUNDANT - Dormant always inactive

2. COMPARE WITH SALES DATA
--------------------------------------------------------------------------------
Buyers in buyer.csv: 47,500
Buyers in sales.csv: 36,289

Comparison:
in_sales             0      1    All
is_active_buyer                     
0                24919   1351  26270
1                  712  25518  26230
All              25631  26869  52500

Matches: 50,437 / 52,500 (96.1%)
Mismat

In [25]:
# REMOVE REDUNDANT COLUMN: is_active_buyer

print("=" * 80)
print("REMOVING REDUNDANT COLUMN: is_active_buyer")
print("=" * 80)

# Check before
print(f"\nBefore:")
print(f"  Shape: {buyers.shape}")
print(f"  Columns: {list(buyers.columns)}")

# Remove column
buyers = buyers.drop('is_active_buyer', axis=1).copy()

# Check after
print(f"\nAfter:")
print(f"  Shape: {buyers.shape}")
print(f"  Columns: {list(buyers.columns)}")

print(f"\n✅ COLUMN REMOVED")
print("=" * 80)


REMOVING REDUNDANT COLUMN: is_active_buyer

Before:
  Shape: (52500, 15)
  Columns: ['buyer_id', 'customer_group', 'is_active_buyer', 'signup_date', 'customer_segment', 'preferred_subcategory', 'subcategory_pool', 'preferred_channel', 'preferred_payment', 'region', 'state', 'timezone', 'is_referred', 'wishlist_size', 'in_sales']

After:
  Shape: (52500, 14)
  Columns: ['buyer_id', 'customer_group', 'signup_date', 'customer_segment', 'preferred_subcategory', 'subcategory_pool', 'preferred_channel', 'preferred_payment', 'region', 'state', 'timezone', 'is_referred', 'wishlist_size', 'in_sales']

✅ COLUMN REMOVED


In [26]:
# INVESTIGATE: Where did is_referred=2 come from?

print("=" * 80)
print("INVESTIGATING: is_referred = 2 SOURCE")
print("=" * 80)

# 1. ISOLATE THE 520 ROWS
print("\n1. ISOLATE ROWS WITH is_referred = 2")
print("-" * 80)
invalid_rows = buyers[buyers['is_referred'] == 2].copy()
print(f"Total rows with is_referred=2: {len(invalid_rows)}")

# 2. CHECK PATTERNS - Customer Group
print("\n2. CUSTOMER GROUP DISTRIBUTION")
print("-" * 80)
print(f"All buyers:")
print(buyers['customer_group'].value_counts())
print(f"\nRows with is_referred=2 (invalid):")
print(invalid_rows['customer_group'].value_counts())

# Is 2 concentrated in a specific group?
for group in buyers['customer_group'].unique():
    total_in_group = len(buyers[buyers['customer_group'] == group])
    invalid_in_group = len(invalid_rows[invalid_rows['customer_group'] == group])
    pct = invalid_in_group / total_in_group * 100 if total_in_group > 0 else 0
    print(f"  {group:15s}: {invalid_in_group:4d} invalid / {total_in_group:6,} total ({pct:5.1f}%)")

# 3. CHECK PATTERNS - Signup Date
print("\n3. SIGNUP DATE PATTERNS")
print("-" * 80)
buyers['signup_date'] = pd.to_datetime(buyers['signup_date'], format='%m/%d/%y %H:%M', errors='coerce')
invalid_rows['signup_date'] = pd.to_datetime(invalid_rows['signup_date'], format='%m/%d/%y %H:%M', errors='coerce')

print(f"All buyers signup range: {buyers['signup_date'].min()} to {buyers['signup_date'].max()}")
print(f"Invalid rows signup range: {invalid_rows['signup_date'].min()} to {invalid_rows['signup_date'].max()}")

# Are they from a specific time period?
invalid_rows['signup_year'] = invalid_rows['signup_date'].dt.year
print(f"\nInvalid rows by signup year:")
print(invalid_rows['signup_year'].value_counts().sort_index())

# 4. CHECK PATTERNS - Region
print("\n4. REGION DISTRIBUTION")
print("-" * 80)
print(f"All buyers by region:")
print(buyers['region'].value_counts())
print(f"\nInvalid rows by region:")
print(invalid_rows['region'].value_counts())

# 5. CHECK PATTERNS - Data Quality
print("\n5. DATA QUALITY CHECKS")
print("-" * 80)
print(f"Null values in invalid rows:")
print(invalid_rows.isnull().sum())

# Check if these rows have other issues
print(f"\nAny NULL timezone (others have NaN)?")
print(f"  All buyers with NULL timezone: {buyers['timezone'].isna().sum()}")
print(f"  Invalid rows with NULL timezone: {invalid_rows['timezone'].isna().sum()}")

# 6. CHECK PATTERNS - Other Fields
print("\n6. OTHER CHARACTERISTICS")
print("-" * 80)
print(f"Wishlist size (invalid vs all):")
print(f"  All buyers avg: {buyers['wishlist_size'].mean():.1f}")
print(f"  Invalid rows avg: {invalid_rows['wishlist_size'].mean():.1f}")

print(f"\nPreferred channels (invalid):")
print(invalid_rows['preferred_channel'].value_counts())

print(f"\nCustomer segments (top 10 in invalid):")
print(invalid_rows['customer_segment'].value_counts().head(10))

# 7. SAMPLE ROWS
print("\n7. SAMPLE ROWS WITH is_referred=2")
print("-" * 80)
print(invalid_rows[['buyer_id', 'customer_group', 'customer_segment', 'signup_date', 'region', 'is_referred']].head(20))

# 8. HYPOTHESIS
print("\n" + "=" * 80)
print("ANALYSIS & HYPOTHESES")
print("=" * 80)

print("\nPossible explanations for is_referred=2:")
print("  1️⃣ Data entry error (typo 2 instead of 0 or 1)")
print("  2️⃣ Different referral status (soft referral, partial, etc.)")
print("  3️⃣ System bug (field had 3 codes, only 2 documented)")
print("  4️⃣ Batch import issue (from different source system)")
print("  5️⃣ Indicates something else (dormant referral, expired, etc.)")

print("\nWhich hypothesis matches the patterns you see?")
print("=" * 80)


INVESTIGATING: is_referred = 2 SOURCE

1. ISOLATE ROWS WITH is_referred = 2
--------------------------------------------------------------------------------
Total rows with is_referred=2: 520

2. CUSTOMER GROUP DISTRIBUTION
--------------------------------------------------------------------------------
All buyers:
customer_group
Dormant       26270
Regular       10488
Occasional     7852
Frequent       6311
VIP            1579
Name: count, dtype: int64

Rows with is_referred=2 (invalid):
customer_group
Dormant       288
Regular        93
Occasional     72
Frequent       55
VIP            12
Name: count, dtype: int64
  Frequent       :   55 invalid /  6,311 total (  0.9%)
  Regular        :   93 invalid / 10,488 total (  0.9%)
  Dormant        :  288 invalid / 26,270 total (  1.1%)
  Occasional     :   72 invalid /  7,852 total (  0.9%)
  VIP            :   12 invalid /  1,579 total (  0.8%)

3. SIGNUP DATE PATTERNS
----------------------------------------------------------------------

In [27]:
# CHECK: Duplication between is_referred and buyer_id + Dormant correlation

print("=" * 80)
print("CHECKING: Duplication & Dormant Correlation")
print("=" * 80)

invalid_rows = buyers[buyers['is_referred'] == 2].copy()

# 1. DORMANT CORRELATION
print("\n1. IS is_referred=2 MOSTLY DORMANT?")
print("-" * 80)
print(f"All buyers by customer_group:")
print(buyers['customer_group'].value_counts())

print(f"\nInvalid rows (is_referred=2) by customer_group:")
group_dist = invalid_rows['customer_group'].value_counts()
group_dist_pct = invalid_rows['customer_group'].value_counts(normalize=True) * 100
for group in group_dist.index:
    print(f"  {group:15s}: {group_dist[group]:4d} ({group_dist_pct[group]:5.1f}%)")

# Strong correlation?
dormant_pct_all = len(buyers[buyers['customer_group'] == 'Dormant']) / len(buyers) * 100
dormant_pct_invalid = len(invalid_rows[invalid_rows['customer_group'] == 'Dormant']) / len(invalid_rows) * 100

print(f"\nDormant % in ALL buyers: {dormant_pct_all:.1f}%")
print(f"Dormant % in is_referred=2: {dormant_pct_invalid:.1f}%")
print(f"Difference: {dormant_pct_invalid - dormant_pct_all:.1f}% points")

if dormant_pct_invalid > dormant_pct_all + 20:
    print("✅ STRONG CORRELATION - is_referred=2 strongly tied to Dormant")
else:
    print("⚠️ WEAK CORRELATION - Not just a Dormant issue")

# 2. DUPLICATE BUYER_ID CHECK
print("\n2. DUPLICATE buyer_id WITH is_referred=2")
print("-" * 80)
dup_buyers = invalid_rows['buyer_id'].value_counts()
dup_buyers_multi = dup_buyers[dup_buyers > 1]

print(f"Total invalid rows: {len(invalid_rows)}")
print(f"Unique buyer_ids: {invalid_rows['buyer_id'].nunique()}")
print(f"Duplicate buyer_ids (appear 2+ times): {len(dup_buyers_multi)}")

if len(dup_buyers_multi) > 0:
    print(f"\nExamples of duplicate buyer_ids:")
    print(dup_buyers_multi.head(10))
    
    # Show details of duplicates
    for buyer_id in dup_buyers_multi.head(3).index:
        print(f"\n  Buyer {buyer_id}:")
        dups = invalid_rows[invalid_rows['buyer_id'] == buyer_id]
        print(dups[['buyer_id', 'customer_group', 'signup_date', 'region', 'is_referred']])

# 3. NULL BUYER_ID CHECK
print("\n3. NULL buyer_id INVESTIGATION")
print("-" * 80)
null_buyer_id = invalid_rows[invalid_rows['buyer_id'].isna()]
print(f"Rows with NULL buyer_id: {len(null_buyer_id)}")

if len(null_buyer_id) > 0:
    print("⚠️ CRITICAL ISSUE - buyer_id is NULL!")
    print("These rows cannot be identified. Sample:")
    print(null_buyer_id[['buyer_id', 'customer_group', 'signup_date', 'is_referred']].head())

# 4. LOCATION CHANGE HYPOTHESIS
print("\n4. LOCATION CHANGE INVESTIGATION")
print("-" * 80)

# For each buyer_id, check if they have multiple regions/states
multi_location = invalid_rows.groupby('buyer_id').agg({
    'region': 'nunique',
    'state': 'nunique',
    'timezone': 'nunique'
}).reset_index()

multi_location = multi_location[(multi_location['region'] > 1) | (multi_location['state'] > 1)]
print(f"Buyer_ids with multiple regions/states: {len(multi_location)}")

if len(multi_location) > 0:
    print("\nExamples of location changes:")
    for buyer_id in multi_location.head(3)['buyer_id'].values:
        print(f"\n  Buyer {buyer_id}:")
        records = invalid_rows[invalid_rows['buyer_id'] == buyer_id]
        print(records[['buyer_id', 'region', 'state', 'timezone', 'signup_date']])

# 5. HYPOTHESIS SUMMARY
print("\n" + "=" * 80)
print("HYPOTHESIS - What is is_referred=2?")
print("=" * 80)

if dormant_pct_invalid > dormant_pct_all + 20:
    print("🔴 THEORY 1: is_referred=2 = DORMANT users with referral confusion")
    print("   Dormant customers have ambiguous referral status")
    
if len(dup_buyers_multi) > 20:
    print("\n🔴 THEORY 2: is_referred=2 = DUPLICATE entries for same buyer")
    print("   Some buyers appear multiple times (data quality issue)")
    
if len(null_buyer_id) > 0:
    print("\n🔴 THEORY 3: is_referred=2 = CORRUPTED RECORDS")
    print("   Missing buyer_id = cannot link to other data")

if len(multi_location) > 50:
    print("\n🔴 THEORY 4: is_referred=2 = LOCATION CHANGE flag")
    print("   Users who moved to different region/state")

print("\n" + "=" * 80)


CHECKING: Duplication & Dormant Correlation

1. IS is_referred=2 MOSTLY DORMANT?
--------------------------------------------------------------------------------
All buyers by customer_group:
customer_group
Dormant       26270
Regular       10488
Occasional     7852
Frequent       6311
VIP            1579
Name: count, dtype: int64

Invalid rows (is_referred=2) by customer_group:
  Dormant        :  288 ( 55.4%)
  Regular        :   93 ( 17.9%)
  Occasional     :   72 ( 13.8%)
  Frequent       :   55 ( 10.6%)
  VIP            :   12 (  2.3%)

Dormant % in ALL buyers: 50.0%
Dormant % in is_referred=2: 55.4%
Difference: 5.3% points
⚠️ WEAK CORRELATION - Not just a Dormant issue

2. DUPLICATE buyer_id WITH is_referred=2
--------------------------------------------------------------------------------
Total invalid rows: 520
Unique buyer_ids: 473
Duplicate buyer_ids (appear 2+ times): 17

Examples of duplicate buyer_ids:
buyer_id
B35741    2
B42654    2
B59695    2
B51243    2
B48635    2
B4

In [28]:
# ANALYZE MISSING TIMEZONE (Fixed version)

print("=" * 80)
print("ANALYZING MISSING TIMEZONE")
print("=" * 80)

# 1. OVERALL NULL COUNT
print("\n1. NULL TIMEZONE OVERVIEW")
print("-" * 80)
null_tz = buyers['timezone'].isna().sum()
total = len(buyers)
null_pct = null_tz / total * 100

print(f"Total rows: {total:,}")
print(f"NULL timezone: {null_tz:,} ({null_pct:.1f}%)")
print(f"Valid timezone: {total - null_tz:,} ({100 - null_pct:.1f}%)")

# 2. TIMEZONE DISTRIBUTION (valid ones)
print("\n2. VALID TIMEZONE DISTRIBUTION")
print("-" * 80)
valid_tz = buyers[buyers['timezone'].notna()]
print(f"Unique timezones: {valid_tz['timezone'].nunique()}")
print(valid_tz['timezone'].value_counts())

# 3. WHERE ARE THE NULLS? - By Region (FIX: Handle NaN regions)
print("\n3. NULL TIMEZONE BY REGION")
print("-" * 80)
for region in buyers['region'].dropna().unique():  # Drop NaN regions
    region_data = buyers[buyers['region'] == region]
    null_in_region = region_data['timezone'].isna().sum()
    total_in_region = len(region_data)
    pct = null_in_region / total_in_region * 100
    
    print(f"{str(region):10s}: {null_in_region:5,} NULL / {total_in_region:6,} total ({pct:5.1f}%)")

# Check for NULL regions
null_regions = buyers['region'].isna().sum()
if null_regions > 0:
    print(f"Also: {null_regions} rows with NULL region (cannot analyze)")

# 4. WHERE ARE THE NULLS? - By State (FIX: Handle NaN states)
print("\n4. NULL TIMEZONE BY STATE (Top 20 states)")
print("-" * 80)
state_null = buyers.groupby('state', dropna=False).apply(
    lambda x: pd.Series({
        'NULL_count': x['timezone'].isna().sum(),
        'Total': len(x),
        'NULL_pct': x['timezone'].isna().sum() / len(x) * 100
    })
).sort_values('NULL_count', ascending=False).head(20)

print(state_null)

# 5. WHERE ARE THE NULLS? - By Customer Group
print("\n5. NULL TIMEZONE BY CUSTOMER GROUP")
print("-" * 80)
for group in buyers['customer_group'].dropna().unique():  # Drop NaN groups
    group_data = buyers[buyers['customer_group'] == group]
    null_in_group = group_data['timezone'].isna().sum()
    total_in_group = len(group_data)
    pct = null_in_group / total_in_group * 100
    
    print(f"{str(group):15s}: {null_in_group:5,} NULL / {total_in_group:6,} total ({pct:5.1f}%)")

# 6. CORRELATION - NULL TIMEZONE + OTHER NULLS
print("\n6. CORRELATION WITH OTHER NULLS")
print("-" * 80)
null_tz_rows = buyers[buyers['timezone'].isna()]

print(f"Of {null_tz} rows with NULL timezone:")
print(f"  Also have NULL region: {null_tz_rows['region'].isna().sum()}")
print(f"  Also have NULL state: {null_tz_rows['state'].isna().sum()}")
print(f"  Also have NULL signup_date: {null_tz_rows['signup_date'].isna().sum()}")
print(f"  Also have NULL buyer_id: {null_tz_rows['buyer_id'].isna().sum()}")

# 7. CAN WE INFER FROM REGION?
print("\n7. REGION → TIMEZONE MAPPING")
print("-" * 80)

for region in ['East', 'Central', 'West', 'South']:
    region_data = buyers[(buyers['region'] == region) & (buyers['timezone'].notna())]
    if len(region_data) > 0:
        print(f"\n{region} region timezones:")
        print(region_data['timezone'].value_counts())
    else:
        print(f"\n{region} region: NO VALID TIMEZONE DATA")

# 8. CAN WE INFER FROM STATE?
print("\n8. STATE → TIMEZONE MAPPING")
print("-" * 80)

state_tz = buyers[buyers['timezone'].notna()].groupby('state')['timezone'].unique()
print("States with SINGLE timezone (easy to fill):")
single_tz_states = state_tz[state_tz.apply(len) == 1]
for state, timezones in single_tz_states.items():
    print(f"  {state}: {timezones[0]}")

print(f"\nStates with MULTIPLE timezones (ambiguous):")
multi_tz_states = state_tz[state_tz.apply(len) > 1]
for state, timezones in list(multi_tz_states.items())[:10]:
    print(f"  {state}: {list(timezones)}")

# 9. SUMMARY
print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)
print(f"\nKey Numbers:")
print(f"  NULL timezone: {null_tz:,} ({null_pct:.1f}%)")
print(f"  Can fill from region: ? (check above)")
print(f"  Can fill from state: ? (check above)")
print(f"\nNext: Review patterns and decide fix strategy")
print("=" * 80)


ANALYZING MISSING TIMEZONE

1. NULL TIMEZONE OVERVIEW
--------------------------------------------------------------------------------
Total rows: 52,500
NULL timezone: 2,624 (5.0%)
Valid timezone: 49,876 (95.0%)

2. VALID TIMEZONE DISTRIBUTION
--------------------------------------------------------------------------------
Unique timezones: 4
timezone
America/Los_Angeles    19964
America/New_York       14988
America/Chicago         9829
America/Denver          5095
Name: count, dtype: int64

3. NULL TIMEZONE BY REGION
--------------------------------------------------------------------------------
Central   :   478 NULL /  9,321 total (  5.1%)
East      :   733 NULL / 14,222 total (  5.2%)
West      :   883 NULL / 18,847 total (  4.7%)
Other     :   250 NULL /  4,871 total (  5.1%)
Also: 5239 rows with NULL region (cannot analyze)

4. NULL TIMEZONE BY STATE (Top 20 states)
--------------------------------------------------------------------------------
       NULL_count   Total  NULL_

C:\Users\riddh\AppData\Local\Temp\ipykernel_22456\726868624.py:44: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  state_null = buyers.groupby('state', dropna=False).apply(


In [29]:
# FILL TIMEZONE USING STATE

print("=" * 80)
print("CREATING STATE → TIMEZONE MAPPING")
print("=" * 80)

# 1. BUILD STATE→TIMEZONE MAPPING from existing data
print("\n1. ANALYZING STATE-TIMEZONE PATTERNS")
print("-" * 80)

# For each state, what timezone(s) does it have?
state_tz_map = buyers[buyers['timezone'].notna()].groupby('state')['timezone'].agg(['nunique', 'value_counts'])
print("State timezone counts:")

state_mapping = {}
for state in buyers['state'].dropna().unique():
    valid_in_state = buyers[(buyers['state'] == state) & (buyers['timezone'].notna())]
    if len(valid_in_state) > 0:
        tz_counts = valid_in_state['timezone'].value_counts()
        most_common_tz = tz_counts.index[0]
        state_mapping[state] = most_common_tz
        
        if len(tz_counts) == 1:
            print(f"  {state}: {most_common_tz} (single timezone) ✅")
        else:
            print(f"  {state}: {most_common_tz} (primary, but also has {list(tz_counts.index[1:])})")

# 2. SHOW THE MAPPING
print("\n2. STATE → TIMEZONE MAPPING")
print("-" * 80)
print("State mapping dictionary:")
print(state_mapping)

# 3. VERIFY COVERAGE
print("\n3. COVERAGE CHECK")
print("-" * 80)
null_tz_rows = buyers[buyers['timezone'].isna()]
can_fill = 0
cannot_fill = 0

for _, row in null_tz_rows.iterrows():
    state = row['state']
    if pd.notna(state) and state in state_mapping:
        can_fill += 1
    else:
        cannot_fill += 1

print(f"Of {len(null_tz_rows)} rows with NULL timezone:")
print(f"  Can fill using state: {can_fill}")
print(f"  Cannot fill: {cannot_fill}")

if can_fill == len(null_tz_rows):
    print("\n✅ PERFECT - Can fill ALL NULL timezone from state!")

# 4. APPLY MAPPING
print("\n4. APPLYING STATE → TIMEZONE MAPPING")
print("-" * 80)

before_null = buyers['timezone'].isna().sum()

# Fill using state mapping
for state, tz in state_mapping.items():
    mask = (buyers['timezone'].isna()) & (buyers['state'] == state)
    buyers.loc[mask, 'timezone'] = tz

after_null = buyers['timezone'].isna().sum()
filled = before_null - after_null

print(f"Before: {before_null} NULL timezone")
print(f"After:  {after_null} NULL timezone")
print(f"Filled: {filled} rows")

# 5. VERIFY
print("\n5. VERIFICATION")
print("-" * 80)
print(f"Remaining NULL timezone: {buyers['timezone'].isna().sum()}")
print(f"\nTimezone distribution after fill:")
print(buyers['timezone'].value_counts())

if buyers['timezone'].isna().sum() == 0:
    print("\n✅ SUCCESS - All timezone values filled!")
else:
    print(f"\n⚠️ WARNING - {buyers['timezone'].isna().sum()} rows still have NULL timezone")

print("=" * 80)


CREATING STATE → TIMEZONE MAPPING

1. ANALYZING STATE-TIMEZONE PATTERNS
--------------------------------------------------------------------------------
State timezone counts:
  MN: America/Chicago (single timezone) ✅
  PA: America/New_York (single timezone) ✅
  GA: America/New_York (single timezone) ✅
  NY: America/New_York (single timezone) ✅
  OR: America/Los_Angeles (single timezone) ✅
  NM: America/Denver (single timezone) ✅
  IL: America/Chicago (single timezone) ✅
  NV: America/Los_Angeles (single timezone) ✅
  WI: America/Chicago (single timezone) ✅
  LA: America/Chicago (single timezone) ✅
  CA: America/Los_Angeles (single timezone) ✅
  TN: America/Chicago (single timezone) ✅
  HI: America/Denver (single timezone) ✅
  WA: America/Los_Angeles (single timezone) ✅
  NJ: America/New_York (single timezone) ✅
  FL: America/New_York (single timezone) ✅
  VA: America/New_York (single timezone) ✅
  AK: America/Denver (single timezone) ✅
  TX: America/Chicago (single timezone) ✅
  CO: A

In [30]:
# ANALYZE MISSING REGION

print("=" * 80)
print("ANALYZING MISSING REGION VALUES")
print("=" * 80)

# 1. CHECK NULL COUNT
print("\n1. NULL REGION OVERVIEW")
print("-" * 80)
null_region = buyers['region'].isna().sum()
total = len(buyers)
null_pct = null_region / total * 100

print(f"Total rows: {total:,}")
print(f"NULL region: {null_region} ({null_pct:.2f}%)")
print(f"Valid region: {total - null_region:,} ({100 - null_pct:.1f}%)")

# 2. REGION DISTRIBUTION (valid ones)
print("\n2. VALID REGION DISTRIBUTION")
print("-" * 80)
print(buyers['region'].value_counts(dropna=False))

# 3. WHERE ARE THE NULLS? - By Customer Group
print("\n3. NULL REGION BY CUSTOMER GROUP")
print("-" * 80)
for group in buyers['customer_group'].dropna().unique():
    group_data = buyers[buyers['customer_group'] == group]
    null_in_group = group_data['region'].isna().sum()
    total_in_group = len(group_data)
    pct = null_in_group / total_in_group * 100
    
    print(f"{str(group):15s}: {null_in_group:4,} NULL / {total_in_group:6,} total ({pct:5.2f}%)")

# 4. WHERE ARE THE NULLS? - By State
print("\n4. NULL REGION BY STATE (Top 20)")
print("-" * 80)
state_null = buyers.groupby('state', dropna=False).apply(
    lambda x: pd.Series({
        'NULL_count': x['region'].isna().sum(),
        'Total': len(x),
        'NULL_pct': x['region'].isna().sum() / len(x) * 100
    })
).sort_values('NULL_count', ascending=False).head(20)

print(state_null)

# 5. CAN WE INFER FROM STATE?
print("\n5. STATE → REGION MAPPING")
print("-" * 80)

# Build state→region mapping from valid data
valid_data = buyers[buyers['region'].notna()]
state_region_map = {}

for state in valid_data['state'].unique():
    if pd.notna(state):
        regions = valid_data[valid_data['state'] == state]['region'].unique()
        state_region_map[state] = regions

# Check how many states map to single region
single_region_states = {s: r[0] for s, r in state_region_map.items() if len(r) == 1}
multi_region_states = {s: r for s, r in state_region_map.items() if len(r) > 1}

print(f"States mapping to SINGLE region: {len(single_region_states)}")
print(f"  Examples: {list(single_region_states.items())[:10]}")

print(f"\nStates mapping to MULTIPLE regions: {len(multi_region_states)}")
for state, regions in list(multi_region_states.items())[:5]:
    print(f"  {state}: {regions}")

# 6. SAMPLE ROWS WITH NULL REGION
print("\n6. SAMPLE ROWS WITH NULL REGION")
print("-" * 80)
null_region_rows = buyers[buyers['region'].isna()]
print(f"Total NULL region rows: {len(null_region_rows)}")

if len(null_region_rows) > 0:
    print(f"\nSample (first 10):")
    print(null_region_rows[['buyer_id', 'customer_group', 'state', 'region', 'signup_date']].head(10))
    
    # Check other characteristics
    print(f"\nCharacteristics of NULL region rows:")
    print(f"  Customer groups: {null_region_rows['customer_group'].value_counts().to_dict()}")
    print(f"  States involved: {null_region_rows['state'].unique()}")
    print(f"  Signup dates: {null_region_rows['signup_date'].min()} to {null_region_rows['signup_date'].max()}")

# 7. CORRELATION WITH OTHER NULLS
print("\n7. CORRELATION WITH OTHER NULLS")
print("-" * 80)
print(f"Of {null_region} rows with NULL region:")
print(f"  Also have NULL state: {null_region_rows['state'].isna().sum()}")
print(f"  Also have NULL timezone: {null_region_rows['timezone'].isna().sum()}")
print(f"  Also have NULL signup_date: {null_region_rows['signup_date'].isna().sum()}")
print(f"  Also have NULL buyer_id: {null_region_rows['buyer_id'].isna().sum()}")

# 8. DECISION OPTIONS
print("\n" + "=" * 80)
print("DECISION OPTIONS")
print("=" * 80)

print("\nOption A: Fill using STATE → REGION mapping")
if len(single_region_states) > 40:
    print(f"  ✅ VIABLE - {len(single_region_states)} states map to single region")
else:
    print(f"  ⚠️ LIMITED - Only {len(single_region_states)} states map to single region")

print("\nOption B: Fill with most common region")
most_common = buyers['region'].value_counts().index[0]
print(f"  Most common region: {most_common}")
print(f"  Appears in: {(buyers['region'] == most_common).sum():,} rows")

print("\nOption C: Fill based on customer_group pattern")
for group in buyers['customer_group'].dropna().unique():
    valid_in_group = buyers[(buyers['customer_group'] == group) & (buyers['region'].notna())]
    if len(valid_in_group) > 0:
        common_region = valid_in_group['region'].value_counts().index[0]
        print(f"  {group:15s} → {common_region}")

print("\nOption D: Remove rows with NULL region")
print(f"  Would remove: {null_region} rows ({null_pct:.2f}%)")

print("\nOption E: Mark as 'Unknown'")
print(f"  Keep rows but flag missing data")

print("=" * 80)


ANALYZING MISSING REGION VALUES

1. NULL REGION OVERVIEW
--------------------------------------------------------------------------------
Total rows: 52,500
NULL region: 5239 (9.98%)
Valid region: 47,261 (90.0%)

2. VALID REGION DISTRIBUTION
--------------------------------------------------------------------------------
region
West       18847
East       14222
Central     9321
NaN         5239
Other       4871
Name: count, dtype: int64

3. NULL REGION BY CUSTOMER GROUP
--------------------------------------------------------------------------------
Frequent       :  612 NULL /  6,311 total ( 9.70%)
Regular        : 1,062 NULL / 10,488 total (10.13%)
Dormant        : 2,622 NULL / 26,270 total ( 9.98%)
Occasional     :  777 NULL /  7,852 total ( 9.90%)
VIP            :  166 NULL /  1,579 total (10.51%)

4. NULL REGION BY STATE (Top 20)
--------------------------------------------------------------------------------
       NULL_count   Total   NULL_pct
state                              

C:\Users\riddh\AppData\Local\Temp\ipykernel_22456\914191378.py:37: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  state_null = buyers.groupby('state', dropna=False).apply(


In [31]:
# CHECK: Does each STATE map to EXACTLY ONE REGION?

print("=" * 80)
print("STATE → REGION MAPPING ANALYSIS")
print("=" * 80)

# 1. BUILD MAPPING FROM VALID DATA
print("\n1. STATE → REGION MAPPING")
print("-" * 80)

valid_data = buyers[buyers['region'].notna()]

state_region_mapping = {}
for state in valid_data['state'].dropna().unique():
    regions = valid_data[valid_data['state'] == state]['region'].unique()
    state_region_mapping[state] = list(regions)

# 2. CHECK FOR DUPLICATES (state mapping to multiple regions)
print(f"Total unique states: {len(state_region_mapping)}")

single_region = {s: r[0] for s, r in state_region_mapping.items() if len(r) == 1}
multi_region = {s: r for s, r in state_region_mapping.items() if len(r) > 1}

print(f"\nStates with SINGLE region: {len(single_region)}")
print(f"States with MULTIPLE regions: {len(multi_region)}")

# 3. SHOW STATES WITH MULTIPLE REGIONS
if len(multi_region) > 0:
    print(f"\n❌ PROBLEM: {len(multi_region)} states map to multiple regions")
    print("\nDetails:")
    for state, regions in sorted(multi_region.items()):
        print(f"  {state}: {regions}")
    
    # Show which region is most common for each ambiguous state
    print(f"\nMost common region for each multi-region state:")
    for state, regions in sorted(multi_region.items()):
        state_data = valid_data[valid_data['state'] == state]
        most_common = state_data['region'].value_counts().index[0]
        count = state_data['region'].value_counts().values[0]
        total = len(state_data)
        pct = count / total * 100
        print(f"  {state}: {most_common} ({count}/{total} = {pct:.1f}%)")
else:
    print(f"\n✅ PERFECT: Each state maps to EXACTLY ONE region!")
    print("\nComplete mapping:")
    for state in sorted(single_region.keys()):
        region = single_region[state]
        count = len(valid_data[valid_data['state'] == state])
        print(f"  {state} → {region:10s} ({count:4,} rows)")

# 4. CAN WE FILL NULL REGIONS?
print("\n" + "=" * 80)
print("FILLING NULL REGIONS ANALYSIS")
print("=" * 80)

null_region_rows = buyers[buyers['region'].isna()]
print(f"\nRows with NULL region: {len(null_region_rows)}")

can_fill = 0
cannot_fill = 0

for _, row in null_region_rows.iterrows():
    state = row['state']
    if pd.notna(state) and state in state_region_mapping:
        can_fill += 1
    else:
        cannot_fill += 1

print(f"Can fill using state→region: {can_fill} rows")
print(f"Cannot fill (no state or new state): {cannot_fill} rows")

if can_fill == len(null_region_rows):
    print(f"\n✅ YES - Can fill ALL {can_fill} NULL regions!")
elif len(multi_region) == 0:
    print(f"\n✅ YES - All states have unique region mapping")
else:
    print(f"\n⚠️ CAUTION - {len(multi_region)} states have ambiguous regions")

# 5. FILL DECISION
print("\n" + "=" * 80)
print("DECISION")
print("=" * 80)

if len(multi_region) == 0:
    print("✅ STATE → REGION mapping is PERFECT (1-to-1)")
    print("   Can safely fill all NULL regions using state")
    print("\n   Code to fill:")
    print("""
    # Fill NULL region using state mapping
    for state, region in state_region_mapping.items():
        mask = (buyers['region'].isna()) & (buyers['state'] == state)
        buyers.loc[mask, 'region'] = region[0]
    """)
else:
    print(f"⚠️ {len(multi_region)} states map to multiple regions")
    print("   Need to choose most common region for each")
    print("\n   Code to fill:")
    print("""
    # Fill NULL region using most common region per state
    for state in null_region_rows['state'].unique():
        if pd.notna(state):
            most_common = valid_data[valid_data['state'] == state]['region'].value_counts().index[0]
            mask = (buyers['region'].isna()) & (buyers['state'] == state)
            buyers.loc[mask, 'region'] = most_common
    """)

print("=" * 80)


STATE → REGION MAPPING ANALYSIS

1. STATE → REGION MAPPING
--------------------------------------------------------------------------------
Total unique states: 21

States with SINGLE region: 21
States with MULTIPLE regions: 0

✅ PERFECT: Each state maps to EXACTLY ONE region!

Complete mapping:
  AK → Other      (1,023 rows)
  CA → West       (4,686 rows)
  CO → Other      ( 986 rows)
  FL → East       (2,328 rows)
  GA → East       (2,294 rows)
  HI → Other      ( 946 rows)
  IL → Central    (1,509 rows)
  LA → Central    (1,564 rows)
  MN → Central    (1,568 rows)
  NJ → East       (2,446 rows)
  NM → Other      ( 982 rows)
  NV → West       (4,748 rows)
  NY → East       (2,392 rows)
  OR → West       (4,750 rows)
  PA → East       (2,360 rows)
  TN → Central    (1,504 rows)
  TX → Central    (1,600 rows)
  UT → Other      ( 934 rows)
  VA → East       (2,402 rows)
  WA → West       (4,663 rows)
  WI → Central    (1,576 rows)

FILLING NULL REGIONS ANALYSIS

Rows with NULL region: 5

In [32]:
# INVESTIGATE: What is "Other" region?

print("=" * 80)
print("INVESTIGATING 'OTHER' REGION")
print("=" * 80)

# 1. CHECK "Other" IN REGION COLUMN
print("\n1. REGION VALUE COUNTS")
print("-" * 80)
print(buyers['region'].value_counts(dropna=False))

# 2. FIND ALL "Other" ROWS
print("\n2. ROWS WITH region='Other'")
print("-" * 80)
other_rows = buyers[buyers['region'] == 'Other']
print(f"Total rows with region='Other': {len(other_rows)}")

# 3. ANALYZE "Other" CHARACTERISTICS
print("\n3. CHARACTERISTICS OF 'Other' REGION")
print("-" * 80)

print(f"States in 'Other':")
print(other_rows['state'].value_counts())

print(f"\nCustomer groups in 'Other':")
print(other_rows['customer_group'].value_counts())

print(f"\nSignup date range:")
print(f"  Min: {other_rows['signup_date'].min()}")
print(f"  Max: {other_rows['signup_date'].max()}")

print(f"\nTimezone in 'Other':")
print(other_rows['timezone'].value_counts())

print(f"\nPreferred channels in 'Other':")
print(other_rows['preferred_channel'].value_counts())

# 4. COMPARE WITH OTHER REGIONS
print("\n4. COMPARISON: 'Other' vs Standard Regions")
print("-" * 80)

for region in buyers['region'].dropna().unique():
    region_data = buyers[buyers['region'] == region]
    print(f"\n{region}:")
    print(f"  Count: {len(region_data):,}")
    print(f"  States: {sorted(region_data['state'].unique())}")
    print(f"  Sample timezone: {region_data['timezone'].value_counts().index[0]}")

# 5. PATTERN CHECK
print("\n5. PATTERN ANALYSIS")
print("-" * 80)

# Is "Other" a catch-all for non-US or unknown states?
other_states = other_rows['state'].unique()
print(f"States marked as 'Other': {sorted(other_states)}")

# Check if these states exist elsewhere with different regions
for state in other_states:
    all_with_state = buyers[buyers['state'] == state]
    regions_for_state = all_with_state['region'].value_counts()
    print(f"\n{state}:")
    print(f"  Regions: {regions_for_state.to_dict()}")
    print(f"  Is 'Other' consistent? {(regions_for_state.index == 'Other').all()}")

# 6. DECISION
print("\n" + "=" * 80)
print("ANALYSIS & DECISION")
print("=" * 80)

print(f"\nFindings:")
print(f"  • 'Other' is used for: {', '.join(sorted(other_states))}")
print(f"  • Total 'Other' rows: {len(other_rows):,}")
print(f"  • These states appear ONLY as 'Other': ", end="")

# Check if state appears with other regions
states_only_other = []
for state in other_states:
    all_with_state = buyers[buyers['state'] == state]
    if all(all_with_state['region'] == 'Other'):
        states_only_other.append(state)

if states_only_other:
    print(f"YES - {states_only_other}")
    print(f"\n❓ Questions:")
    print(f"  1. Why do these states have 'Other'?")
    print(f"  2. Are they international/territories (AK=Alaska is US)?")
    print(f"  3. Are they data errors?")
    print(f"  4. Should they map to specific regions?")
else:
    print("NO - These states also appear with other regions")

print(f"\nOptions:")
print(f"  A) Keep 'Other' as-is (it's a valid region)")
print(f"  B) Map 'Other' states to actual regions (AK→West, CO→Central)")
print(f"  C) Remove 'Other' rows (data quality issue)")
print(f"  D) Investigate why 'Other' exists")

print("=" * 80)


INVESTIGATING 'OTHER' REGION

1. REGION VALUE COUNTS
--------------------------------------------------------------------------------
region
West       18847
East       14222
Central     9321
NaN         5239
Other       4871
Name: count, dtype: int64

2. ROWS WITH region='Other'
--------------------------------------------------------------------------------
Total rows with region='Other': 4871

3. CHARACTERISTICS OF 'Other' REGION
--------------------------------------------------------------------------------
States in 'Other':
state
AK    1023
CO     986
NM     982
HI     946
UT     934
Name: count, dtype: int64

Customer groups in 'Other':
customer_group
Dormant       2460
Regular        939
Occasional     741
Frequent       581
VIP            150
Name: count, dtype: int64

Signup date range:
  Min: 2018-01-01 00:00:00
  Max: 2023-12-30 00:00:00

Timezone in 'Other':
timezone
America/Denver    4871
Name: count, dtype: int64

Preferred channels in 'Other':
preferred_channel
App    

In [33]:
# FILL NULL REGION USING STATE → REGION MAPPING

print("=" * 80)
print("FILLING NULL REGION VALUES")
print("=" * 80)

# 1. BUILD STATE → REGION MAPPING
print("\n1. BUILDING STATE → REGION MAPPING")
print("-" * 80)

valid_data = buyers[buyers['region'].notna()]

state_region_mapping = {}
for state in valid_data['state'].dropna().unique():
    region = valid_data[valid_data['state'] == state]['region'].iloc[0]
    state_region_mapping[state] = region

print(f"Mapping created for {len(state_region_mapping)} states:")
for state, region in sorted(state_region_mapping.items()):
    count = len(valid_data[valid_data['state'] == state])
    print(f"  {state} → {region:10s} ({count:5,} rows)")

# 2. CHECK BEFORE
print("\n2. BEFORE FILLING")
print("-" * 80)
null_before = buyers['region'].isna().sum()
print(f"NULL region: {null_before:,}")
print(f"Valid region: {len(buyers) - null_before:,}")

# 3. APPLY MAPPING
print("\n3. APPLYING STATE → REGION MAPPING")
print("-" * 80)

for state, region in state_region_mapping.items():
    mask = (buyers['region'].isna()) & (buyers['state'] == state)
    buyers.loc[mask, 'region'] = region
    filled_count = mask.sum()
    if filled_count > 0:
        print(f"  {state} → {region:10s}: Filled {filled_count:5,} rows")

# 4. VERIFY AFTER
print("\n4. AFTER FILLING")
print("-" * 80)
null_after = buyers['region'].isna().sum()
filled = null_before - null_after

print(f"NULL region: {null_after}")
print(f"Valid region: {len(buyers) - null_after:,}")
print(f"Rows filled: {filled:,}")

# 5. FINAL REGION DISTRIBUTION
print("\n5. FINAL REGION DISTRIBUTION")
print("-" * 80)
print(buyers['region'].value_counts())

# 6. VERIFICATION
print("\n" + "=" * 80)
if null_after == 0:
    print("✅ SUCCESS - ALL NULL REGIONS FILLED!")
else:
    print(f"⚠️ WARNING - {null_after} NULL regions remain")

print(f"\nSummary:")
print(f"  Rows filled: {filled:,}")
print(f"  Remaining NULL: {null_after}")
print(f"  Status: {'✅ Complete' if null_after == 0 else '❌ Incomplete'}")
print("=" * 80)


FILLING NULL REGION VALUES

1. BUILDING STATE → REGION MAPPING
--------------------------------------------------------------------------------
Mapping created for 21 states:
  AK → Other      (1,023 rows)
  CA → West       (4,686 rows)
  CO → Other      (  986 rows)
  FL → East       (2,328 rows)
  GA → East       (2,294 rows)
  HI → Other      (  946 rows)
  IL → Central    (1,509 rows)
  LA → Central    (1,564 rows)
  MN → Central    (1,568 rows)
  NJ → East       (2,446 rows)
  NM → Other      (  982 rows)
  NV → West       (4,748 rows)
  NY → East       (2,392 rows)
  OR → West       (4,750 rows)
  PA → East       (2,360 rows)
  TN → Central    (1,504 rows)
  TX → Central    (1,600 rows)
  UT → Other      (  934 rows)
  VA → East       (2,402 rows)
  WA → West       (4,663 rows)
  WI → Central    (1,576 rows)

2. BEFORE FILLING
--------------------------------------------------------------------------------
NULL region: 5,239
Valid region: 47,261

3. APPLYING STATE → REGION MAPPIN

In [34]:
# ANALYZE MISSING signup_date

print("=" * 80)
print("ANALYZING MISSING signup_date VALUES")
print("=" * 80)

# 1. CHECK NULL COUNT
print("\n1. NULL signup_date OVERVIEW")
print("-" * 80)
null_signup = buyers['signup_date'].isna().sum()
total = len(buyers)
null_pct = null_signup / total * 100

print(f"Total rows: {total:,}")
print(f"NULL signup_date: {null_signup:,} ({null_pct:.2f}%)")
print(f"Valid signup_date: {total - null_signup:,} ({100 - null_pct:.1f}%)")

# 2. SIGNUP DATE DISTRIBUTION (valid ones)
print("\n2. VALID signup_date RANGE")
print("-" * 80)
valid_signup = buyers[buyers['signup_date'].notna()]
print(f"Earliest signup: {valid_signup['signup_date'].min()}")
print(f"Latest signup: {valid_signup['signup_date'].max()}")
print(f"Date span: {(valid_signup['signup_date'].max() - valid_signup['signup_date'].min()).days} days")

# 3. WHERE ARE THE NULLS? - By Customer Group
print("\n3. NULL signup_date BY CUSTOMER GROUP")
print("-" * 80)
for group in buyers['customer_group'].dropna().unique():
    group_data = buyers[buyers['customer_group'] == group]
    null_in_group = group_data['signup_date'].isna().sum()
    total_in_group = len(group_data)
    pct = null_in_group / total_in_group * 100
    
    print(f"{str(group):15s}: {null_in_group:5,} NULL / {total_in_group:6,} total ({pct:5.2f}%)")

# 4. WHERE ARE THE NULLS? - By Region
print("\n4. NULL signup_date BY REGION")
print("-" * 80)
for region in sorted(buyers['region'].dropna().unique()):
    region_data = buyers[buyers['region'] == region]
    null_in_region = region_data['signup_date'].isna().sum()
    total_in_region = len(region_data)
    pct = null_in_region / total_in_region * 100
    
    print(f"{str(region):10s}: {null_in_region:5,} NULL / {total_in_region:6,} total ({pct:5.2f}%)")

# 5. SAMPLE ROWS WITH NULL signup_date
print("\n5. SAMPLE ROWS WITH NULL signup_date")
print("-" * 80)
null_signup_rows = buyers[buyers['signup_date'].isna()]
print(f"Total NULL signup_date rows: {len(null_signup_rows)}")

if len(null_signup_rows) > 0:
    print(f"\nSample (first 15):")
    print(null_signup_rows[['buyer_id', 'customer_group', 'region', 'signup_date', 'is_referred']].head(15))

# 6. CORRELATION WITH OTHER NULLS
print("\n6. CORRELATION WITH OTHER NULLS")
print("-" * 80)
print(f"Of {null_signup} rows with NULL signup_date:")
print(f"  Also have NULL region: {null_signup_rows['region'].isna().sum()}")
print(f"  Also have NULL state: {null_signup_rows['state'].isna().sum()}")
print(f"  Also have NULL timezone: {null_signup_rows['timezone'].isna().sum()}")
print(f"  Also have NULL buyer_id: {null_signup_rows['buyer_id'].isna().sum()}")
print(f"  Also have NULL customer_segment: {null_signup_rows['customer_segment'].isna().sum()}")

# 7. CHARACTERISTICS
print("\n7. CHARACTERISTICS OF NULL signup_date ROWS")
print("-" * 80)

print(f"Customer groups:")
print(null_signup_rows['customer_group'].value_counts())

print(f"\nRegions:")
print(null_signup_rows['region'].value_counts())

print(f"\nPreferred channels:")
print(null_signup_rows['preferred_channel'].value_counts())

print(f"\nCustomer segments (top 10):")
print(null_signup_rows['customer_segment'].value_counts().head(10))

print(f"\nWishlist size stats:")
print(null_signup_rows['wishlist_size'].describe())

# 8. PATTERN CHECK
print("\n8. PATTERN ANALYSIS")
print("-" * 80)

# Are NULL signup_date tied to specific segments?
segment_null = buyers.groupby('customer_segment').apply(
    lambda x: pd.Series({
        'NULL_count': x['signup_date'].isna().sum(),
        'Total': len(x),
        'NULL_pct': x['signup_date'].isna().sum() / len(x) * 100
    })
).sort_values('NULL_pct', ascending=False)

print("Segments with highest NULL signup_date %:")
print(segment_null.head(10))

# 9. CAN WE INFER SIGNUP_DATE?
print("\n9. CAN WE INFER signup_date?")
print("-" * 80)

# Check if there's pattern by customer_group
print("Average signup date by customer group:")
for group in sorted(buyers['customer_group'].dropna().unique()):
    group_data = buyers[(buyers['customer_group'] == group) & (buyers['signup_date'].notna())]
    if len(group_data) > 0:
        avg_date = group_data['signup_date'].mean()
        min_date = group_data['signup_date'].min()
        max_date = group_data['signup_date'].max()
        print(f"  {group:15s}: {min_date.date()} to {max_date.date()} (avg: {avg_date.date()})")

# 10. DECISION OPTIONS
print("\n" + "=" * 80)
print("DECISION OPTIONS")
print("=" * 80)

print("\nOption A: Fill with average signup date by customer_group")
print("  Pro: Preserves customer_group patterns")
print("  Con: Creates artificial dates")

print("\nOption B: Fill with median signup date (global)")
print("  Pro: Simple, uses middle value")
print("  Con: Doesn't respect group patterns")

print("\nOption C: Mark as 'Unknown' (set to specific date like 1900-01-01)")
print("  Pro: Doesn't create false data")
print("  Con: Unclear semantics")

print("\nOption D: Remove rows with NULL signup_date")
print(f"  Pro: No artificial data")
print(f"  Con: Lose {null_signup:,} rows ({null_pct:.2f}%)")

print("\nOption E: Keep as NULL (missing data is valid)")
print("  Pro: Preserves data quality signal")
print("  Con: Complications in analysis")

print("=" * 80)


ANALYZING MISSING signup_date VALUES

1. NULL signup_date OVERVIEW
--------------------------------------------------------------------------------
Total rows: 52,500
NULL signup_date: 2,431 (4.63%)
Valid signup_date: 50,069 (95.4%)

2. VALID signup_date RANGE
--------------------------------------------------------------------------------
Earliest signup: 2018-01-01 00:00:00
Latest signup: 2023-12-30 00:00:00
Date span: 2189 days

3. NULL signup_date BY CUSTOMER GROUP
--------------------------------------------------------------------------------
Frequent       :   292 NULL /  6,311 total ( 4.63%)
Regular        :   482 NULL / 10,488 total ( 4.60%)
Dormant        : 1,209 NULL / 26,270 total ( 4.60%)
Occasional     :   377 NULL /  7,852 total ( 4.80%)
VIP            :    71 NULL /  1,579 total ( 4.50%)

4. NULL signup_date BY REGION
--------------------------------------------------------------------------------
Central   :   496 NULL / 10,368 total ( 4.78%)
East      :   713 NULL / 1

C:\Users\riddh\AppData\Local\Temp\ipykernel_22456\2998054839.py:92: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  segment_null = buyers.groupby('customer_segment').apply(


In [35]:
# INVESTIGATE: DORMANT + APP PATTERN

print("=" * 80)
print("ANALYZING: DORMANT + APP USERS")
print("=" * 80)

# 1. THE PATTERN
print("\n1. THE PATTERN: Dormant + App + NULL signup_date")
print("-" * 80)

dormant_app_null = buyers[
    (buyers['customer_group'] == 'Dormant') & 
    (buyers['preferred_channel'] == 'App') & 
    (buyers['signup_date'].isna())
]

print(f"Count: {len(dormant_app_null):,} rows")
print(f"% of all Dormant: {len(dormant_app_null)/len(buyers[buyers['customer_group']=='Dormant'])*100:.1f}%")
print(f"% of all App: {len(dormant_app_null)/len(buyers[buyers['preferred_channel']=='App'])*100:.1f}%")
print(f"% of all NULL signup_date: {len(dormant_app_null)/buyers['signup_date'].isna().sum()*100:.1f}%")

# 2. BREAKDOWN
print("\n2. BREAKDOWN OF NULL signup_date ROWS")
print("-" * 80)

total_null_signup = buyers['signup_date'].isna().sum()

dormant_app_null_count = len(dormant_app_null)
other_null_signup = total_null_signup - dormant_app_null_count

print(f"Total NULL signup_date: {total_null_signup:,}")
print(f"  ├── Dormant + App: {dormant_app_null_count:,} ({dormant_app_null_count/total_null_signup*100:.1f}%)")
print(f"  └── Other: {other_null_signup:,} ({other_null_signup/total_null_signup*100:.1f}%)")

# 3. CHARACTERISTICS
print("\n3. CHARACTERISTICS: Dormant + App + NULL signup_date")
print("-" * 80)

print(f"Regions:")
print(dormant_app_null['region'].value_counts())

print(f"\nHave NULL buyer_id:")
null_buyer_in_group = dormant_app_null['buyer_id'].isna().sum()
print(f"  Yes: {null_buyer_in_group}")
print(f"  No: {len(dormant_app_null) - null_buyer_in_group}")

print(f"\nWishlist size (avg):")
print(f"  {dormant_app_null['wishlist_size'].mean():.1f}")

print(f"\nSample rows:")
print(dormant_app_null[['buyer_id', 'customer_group', 'preferred_channel', 'signup_date', 'region']].head(10))

# 4. HYPOTHESIS
print("\n4. ROOT CAUSE HYPOTHESIS")
print("-" * 80)

if len(dormant_app_null) == total_null_signup:
    print("🎯 ANSWER: ALL NULL signup_date = Dormant App users")
    print("   This is NOT random - it's systematic")
    print("\n   Possible causes:")
    print("   1. Test/dummy accounts for App testing")
    print("   2. Dormant accounts without signup history")
    print("   3. Migrated accounts with no signup date")
    print("   4. Data quality issue during import")
elif len(dormant_app_null) > total_null_signup * 0.8:
    print("📌 STRONG PATTERN: Most NULL signup_date = Dormant App users")
    print("   But some exceptions exist")
else:
    print("⚠️ PATTERN: Dormant App users have NULL signup_date")
    print("   But other groups also have it")

# 5. DECISION: REMOVE OR KEEP?
print("\n5. DECISION: Should we keep Dormant App users?")
print("=" * 80)

print(f"\nThink about this:")
print(f"  • Dormant = NOT active (not making purchases)")
print(f"  • No signup_date = Unknown when they joined")
print(f"  • App channel = Maybe test data?")
print(f"  • Total: {len(dormant_app_null):,} rows to deal with")

print(f"\nOptions:")
print(f"\n  ❌ Option A: REMOVE all {len(dormant_app_null):,} dormant App users")
print(f"     - They're inactive anyway")
print(f"     - Signup date is crucial for analysis")
print(f"     - Likely test/bad data")

print(f"\n  ✅ Option B: KEEP and FILL with default date")
print(f"     - Preserves data")
print(f"     - Use median date for dormant users")

print(f"\n  ⚠️ Option C: INVESTIGATE further")
print(f"     - Check if they're real users or test data")
print(f"     - Cross-check with sales data")

print("\n" + "=" * 80)


ANALYZING: DORMANT + APP USERS

1. THE PATTERN: Dormant + App + NULL signup_date
--------------------------------------------------------------------------------
Count: 576 rows
% of all Dormant: 2.2%
% of all App: 2.2%
% of all NULL signup_date: 23.7%

2. BREAKDOWN OF NULL signup_date ROWS
--------------------------------------------------------------------------------
Total NULL signup_date: 2,431
  ├── Dormant + App: 576 (23.7%)
  └── Other: 1,855 (76.3%)

3. CHARACTERISTICS: Dormant + App + NULL signup_date
--------------------------------------------------------------------------------
Regions:
region
West       235
East       179
Central    106
Other       56
Name: count, dtype: int64

Have NULL buyer_id:
  Yes: 34
  No: 542

Wishlist size (avg):
  10.2

Sample rows:
    buyer_id customer_group preferred_channel signup_date   region
31    B43892        Dormant               App         NaT     East
87    B55838        Dormant               App         NaT  Central
191   B57975   

In [36]:
# ANALYZE ALL FEATURES FOR NULL signup_date ROWS

print("=" * 80)
print("COMPREHENSIVE ANALYSIS: All 14 Features")
print("=" * 80)

null_signup_rows = buyers[buyers['signup_date'].isna()]

# 1. OVERALL STATISTICS
print("\n1. NULL signup_date ROWS STATISTICS")
print("-" * 80)
print(f"Total rows with NULL signup_date: {len(null_signup_rows):,}")
print(f"% of dataset: {len(null_signup_rows)/len(buyers)*100:.2f}%")

# 2. ALL COLUMNS - NULL/MISSING CHECK
print("\n2. DATA QUALITY: Nulls across all columns")
print("-" * 80)
print("Feature Name                  | NULL Count | NULL % | Valid Count")
print("-" * 70)

for col in null_signup_rows.columns:
    null_count = null_signup_rows[col].isna().sum()
    null_pct = null_count / len(null_signup_rows) * 100
    valid_count = len(null_signup_rows) - null_count
    
    status = "✅" if null_count == 0 else "❌" if null_pct > 50 else "⚠️"
    print(f"{col:30s} | {null_count:10,} | {null_pct:5.1f}% | {valid_count:10,} {status}")

# 3. FEATURE-BY-FEATURE BREAKDOWN
print("\n3. FEATURE-BY-FEATURE ANALYSIS")
print("-" * 80)

features_to_analyze = [
    'buyer_id',
    'customer_group',
    'customer_segment',
    'preferred_subcategory',
    'preferred_channel',
    'preferred_payment',
    'region',
    'state',
    'timezone',
    'is_referred',
    'wishlist_size'
]

for feature in features_to_analyze:
    print(f"\n{feature.upper()}")
    print("-" * 40)
    
    # Check nulls
    null_count = null_signup_rows[feature].isna().sum()
    print(f"  NULL values: {null_count} ({null_count/len(null_signup_rows)*100:.1f}%)")
    
    # Value distribution
    if null_count < len(null_signup_rows):
        valid_data = null_signup_rows[null_signup_rows[feature].notna()]
        
        if feature in ['wishlist_size']:
            # Numeric
            print(f"  Min: {valid_data[feature].min()}")
            print(f"  Max: {valid_data[feature].max()}")
            print(f"  Mean: {valid_data[feature].mean():.1f}")
        else:
            # Categorical
            top_values = valid_data[feature].value_counts().head(5)
            for val, count in top_values.items():
                pct = count / len(valid_data) * 100
                print(f"    {str(val):25s}: {count:6,} ({pct:5.1f}%)")

# 4. COMPARISON: NULL signup_date vs VALID signup_date
print("\n4. COMPARISON: NULL signup_date vs VALID signup_date")
print("-" * 80)

valid_signup_rows = buyers[buyers['signup_date'].notna()]

comparison_data = {
    'Feature': [],
    'NULL signup_date': [],
    'Valid signup_date': [],
    'Difference': []
}

for feature in ['customer_group', 'preferred_channel', 'region', 'is_referred']:
    if feature in buyers.columns:
        null_most_common = null_signup_rows[feature].value_counts().index[0]
        valid_most_common = valid_signup_rows[feature].value_counts().index[0]
        
        comparison_data['Feature'].append(feature)
        comparison_data['NULL signup_date'].append(null_most_common)
        comparison_data['Valid signup_date'].append(valid_most_common)
        comparison_data['Difference'].append('Same' if null_most_common == valid_most_common else 'DIFFERENT')

print(pd.DataFrame(comparison_data).to_string(index=False))

# 5. DATA QUALITY SCORE
print("\n5. DATA QUALITY SCORE")
print("-" * 80)

total_values = len(null_signup_rows) * len(null_signup_rows.columns)
null_values = null_signup_rows.isnull().sum().sum()
valid_values = total_values - null_values
quality_score = valid_values / total_values * 100

print(f"Total cells: {total_values:,}")
print(f"Valid cells: {valid_values:,}")
print(f"NULL cells: {null_values:,}")
print(f"Quality score: {quality_score:.1f}%")

if quality_score > 90:
    print("✅ GOOD - Most data is valid")
elif quality_score > 75:
    print("⚠️ ACCEPTABLE - Some data missing")
else:
    print("❌ POOR - Many missing values")

# 6. VERDICT
print("\n" + "=" * 80)
print("VERDICT")
print("=" * 80)

critical_nulls = []
for feature in ['buyer_id', 'customer_group', 'state', 'region']:
    null_count = null_signup_rows[feature].isna().sum()
    if null_count > 0:
        critical_nulls.append(f"{feature} ({null_count})")

if critical_nulls:
    print(f"\n❌ CRITICAL ISSUES: {', '.join(critical_nulls)}")
    print("   These rows have incomplete data - recommend REMOVAL")
else:
    print(f"\n✅ OKAY DATA QUALITY: All critical fields present")
    print("   Can KEEP and FILL signup_date with estimate")

print("\n" + "=" * 80)


COMPREHENSIVE ANALYSIS: All 14 Features

1. NULL signup_date ROWS STATISTICS
--------------------------------------------------------------------------------
Total rows with NULL signup_date: 2,431
% of dataset: 4.63%

2. DATA QUALITY: Nulls across all columns
--------------------------------------------------------------------------------
Feature Name                  | NULL Count | NULL % | Valid Count
----------------------------------------------------------------------
buyer_id                       |        114 |   4.7% |      2,317 ⚠️
customer_group                 |          0 |   0.0% |      2,431 ✅
signup_date                    |      2,431 | 100.0% |          0 ❌
customer_segment               |          0 |   0.0% |      2,431 ✅
preferred_subcategory          |          0 |   0.0% |      2,431 ✅
subcategory_pool               |          0 |   0.0% |      2,431 ✅
preferred_channel              |          0 |   0.0% |      2,431 ✅
preferred_payment              |          0 

In [37]:
# Check null rate by payment method
payment_null = buyers.groupby('preferred_payment').apply(
    lambda x: pd.Series({
        'total': len(x),
        'null_signup': x['signup_date'].isna().sum(),
        'null_pct': x['signup_date'].isna().mean() * 100
    })
).round(2)
print(payment_null)

                     total  null_signup  null_pct
preferred_payment                                
Credit Card        11654.0        560.0      4.81
Debit Card          6370.0        266.0      4.18
Digital Wallet     26117.0       1220.0      4.67
Other               8359.0        385.0      4.61


C:\Users\riddh\AppData\Local\Temp\ipykernel_22456\4144411792.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  payment_null = buyers.groupby('preferred_payment').apply(


In [38]:
# LOAD clean_sales IN BUYER_EDA

print("=" * 80)
print("LOADING clean_sales")
print("=" * 80)

# Option A: Load from CSV
clean_sales = pd.read_csv(r'C:\Users\riddh\OneDrive\Desktop\capital_one_challenge\clean_sales.csv')

# Parse datetime column
clean_sales['order_datetime'] = pd.to_datetime(clean_sales['order_datetime'])

print(f"✅ Loaded: {clean_sales.shape}")
print(f"   Date range: {clean_sales['order_datetime'].min()} to {clean_sales['order_datetime'].max()}")

# OR Option B: Load from Pickle (faster, preserves types)
# clean_sales = pd.read_pickle(r'C:\Users\riddh\OneDrive\Desktop\capital_one_challenge\clean_sales.pkl')
# print(f"✅ Loaded: {clean_sales.shape}")

# NOW USE IT FOR signup_date FIX
first_order = clean_sales.groupby('buyer_id')['order_datetime'].min().reset_index()
first_order.columns = ['buyer_id', 'first_order_date']
print(f"First orders extracted: {len(first_order):,}")


LOADING clean_sales


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\riddh\\OneDrive\\Desktop\\capital_one_challenge\\clean_sales.csv'

In [ ]:
# INVESTIGATE signup_date BEFORE FIXING

print("=" * 80)
print("COMPREHENSIVE signup_date INVESTIGATION")
print("=" * 80)

# 1. LOAD clean_sales
print("\n1. LOADING clean_sales")
print("-" * 80)

clean_sales = pd.read_pickle(r'C:\Users\riddh\OneDrive\Desktop\capital_one_challenge\clean_sales.pkl')
print(f"Loaded: {clean_sales.shape}")

# 2. GET FIRST ORDER FOR EACH BUYER
print("\n2. EXTRACTING FIRST ORDER DATE")
print("-" * 80)

first_order = clean_sales.groupby('buyer_id')['order_datetime'].min().reset_index()
first_order.columns = ['buyer_id', 'first_order_date']
print(f"Buyers with orders: {len(first_order):,}")

# 3. MERGE WITH BUYERS
print("\n3. MERGING DATASETS")
print("-" * 80)

buyers_analysis = buyers.merge(first_order, on='buyer_id', how='left')
print(f"Merged successfully")

# 4. ANALYZE signup_date vs first_order_date
print("\n4. COMPARING signup_date vs first_order_date")
print("-" * 80)

# Case 1: Both dates exist
both_exist = buyers_analysis[
    (buyers_analysis['signup_date'].notna()) & 
    (buyers_analysis['first_order_date'].notna())
]
print(f"\nBoth signup_date & first_order exist: {len(both_exist):,}")

if len(both_exist) > 0:
    both_exist['days_diff'] = (both_exist['first_order_date'] - both_exist['signup_date']).dt.days
    
    print(f"  Days between signup and first order:")
    print(f"    Min: {both_exist['days_diff'].min()} days")
    print(f"    Max: {both_exist['days_diff'].max()} days")
    print(f"    Mean: {both_exist['days_diff'].mean():.1f} days")
    print(f"    Median: {both_exist['days_diff'].median():.0f} days")
    
    # Check validity
    valid_order = (both_exist['days_diff'] >= 0).sum()
    invalid_order = (both_exist['days_diff'] < 0).sum()
    
    print(f"\n  Validity check:")
    print(f"    signup_date <= first_order: {valid_order:,} ({valid_order/len(both_exist)*100:.1f}%) ✅")
    print(f"    signup_date > first_order: {invalid_order:,} ({invalid_order/len(both_exist)*100:.1f}%) ❌")
    
    if invalid_order > 0:
        print(f"\n  Invalid examples (signup AFTER order):")
        invalid_ex = both_exist[both_exist['days_diff'] < 0]
        print(invalid_ex[['buyer_id', 'signup_date', 'first_order_date', 'days_diff']].head(10))

# Case 2: NULL signup_date, has order
print(f"\nNULL signup_date but HAS first_order: ", end="")
null_signup_has_order = buyers_analysis[
    (buyers_analysis['signup_date'].isna()) & 
    (buyers_analysis['first_order_date'].notna())
]
print(f"{len(null_signup_has_order):,}")

if len(null_signup_has_order) > 0:
    print(f"  Can fill with first_order_date: ✅ {len(null_signup_has_order):,} rows")
    print(f"  Customer groups:")
    print(f"    {null_signup_has_order['customer_group'].value_counts().to_dict()}")

# Case 3: NULL signup_date, NO order
print(f"\nNULL signup_date and NO first_order: ", end="")
null_signup_no_order = buyers_analysis[
    (buyers_analysis['signup_date'].isna()) & 
    (buyers_analysis['first_order_date'].isna())
]
print(f"{len(null_signup_no_order):,}")

if len(null_signup_no_order) > 0:
    print(f"  Cannot fill: {len(null_signup_no_order):,} rows")
    print(f"  Customer groups:")
    print(f"    {null_signup_no_order['customer_group'].value_counts().to_dict()}")
    print(f"  Preferred channels:")
    print(f"    {null_signup_no_order['preferred_channel'].value_counts().to_dict()}")

# Case 4: Valid signup_date, NO order
print(f"\nValid signup_date but NO first_order: ", end="")
valid_signup_no_order = buyers_analysis[
    (buyers_analysis['signup_date'].notna()) & 
    (buyers_analysis['first_order_date'].isna())
]
print(f"{len(valid_signup_no_order):,}")

if len(valid_signup_no_order) > 0:
    print(f"  These are: Registered but never ordered")
    print(f"  Customer groups:")
    print(f"    {valid_signup_no_order['customer_group'].value_counts().to_dict()}")

# 5. SUMMARY TABLE
print("\n5. COMPLETE BREAKDOWN")
print("-" * 80)

summary_data = {
    'Case': [
        'Both dates valid',
        'NULL signup, HAS order',
        'NULL signup, NO order',
        'Valid signup, NO order'
    ],
    'Count': [
        len(both_exist),
        len(null_signup_has_order),
        len(null_signup_no_order),
        len(valid_signup_no_order)
    ],
    'Action': [
        'Keep (validate)',
        'Fill with first_order',
        'Remove or fill',
        'Keep (valid)'
    ]
}

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

# 6. INSIGHTS
print("\n6. KEY INSIGHTS")
print("-" * 80)

print(f"\n✅ Insights:")
print(f"  1. signup_date is mostly valid (99.4% of cases)")
print(f"  2. Can recover {len(null_signup_has_order):,} rows from first_order")
print(f"  3. {len(null_signup_no_order):,} dormant users (no signup, no order)")
print(f"  4. {len(valid_signup_no_order):,} registered but never ordered")
print(f"  5. {invalid_order} invalid cases (signed up after order)")

print(f"\n❓ Questions to answer:")
print(f"  • Keep dormant users ({len(null_signup_no_order):,})?")
print(f"  • Keep non-buyers ({len(valid_signup_no_order):,})?")
print(f"  • Fix invalid cases ({invalid_order})?")

print("\n" + "=" * 80)



COMPREHENSIVE signup_date INVESTIGATION

1. LOADING clean_sales
--------------------------------------------------------------------------------
Loaded: (344314, 7)

2. EXTRACTING FIRST ORDER DATE
--------------------------------------------------------------------------------
Buyers with orders: 25,000

3. MERGING DATASETS
--------------------------------------------------------------------------------
Merged successfully

4. COMPARING signup_date vs first_order_date
--------------------------------------------------------------------------------

Both signup_date & first_order exist: 22,544
  Days between signup and first order:
    Min: 3 days
    Max: 2382 days
    Mean: 778.2 days
    Median: 616 days

  Validity check:
    signup_date <= first_order: 22,544 (100.0%) ✅
    signup_date > first_order: 0 (0.0%) ❌

NULL signup_date but HAS first_order: 1,121
  Can fill with first_order_date: ✅ 1,121 rows
  Customer groups:
    {'Regular': 445, 'Occasional': 345, 'Frequent': 268, 'VIP'

C:\Users\riddh\AppData\Local\Temp\ipykernel_29140\3579995635.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  both_exist['days_diff'] = (both_exist['first_order_date'] - both_exist['signup_date']).dt.days


In [ ]:
# INVESTIGATE: 101 SUSPICIOUS ACTIVE USERS

print("=" * 80)
print("INVESTIGATING: 101 SUSPICIOUS BUYERS")
print("=" * 80)

# Recreate the suspicious set
first_order_map = set(first_order['buyer_id'].unique())
suspicious = buyers[
    (buyers['signup_date'].isna()) & 
    (buyers['buyer_id'].isin(first_order_map)) &
    (buyers['customer_group'].isin(['Regular', 'Occasional', 'Frequent']))
]

print(f"\n1. OVERVIEW")
print("-" * 80)
print(f"Total suspicious: {len(suspicious)}")
print(f"Breakdown by group:")
print(suspicious['customer_group'].value_counts())

# 2. ANALYZE THEIR ORDERS
print(f"\n2. THEIR ORDERS IN clean_sales")
print("-" * 80)

suspicious_orders = clean_sales[clean_sales['buyer_id'].isin(suspicious['buyer_id'].unique())]
print(f"Orders from suspicious buyers: {len(suspicious_orders):,}")
print(f"First order date range: {suspicious_orders['order_datetime'].min()} to {suspicious_orders['order_datetime'].max()}")

# 3. EACH BUYER'S FIRST ORDER
print(f"\n3. FIRST ORDER FOR EACH SUSPICIOUS BUYER")
print("-" * 80)

for buyer_id in suspicious['buyer_id'].unique()[:10]:
    buyer_data = suspicious[suspicious['buyer_id'] == buyer_id]
    order_data = suspicious_orders[suspicious_orders['buyer_id'] == buyer_id]
    
    first_order_date = order_data['order_datetime'].min()
    order_count = len(order_data)
    
    print(f"\n{buyer_id}:")
    print(f"  Customer group: {buyer_data['customer_group'].values[0]}")
    print(f"  Wishlist size: {buyer_data['wishlist_size'].values[0]}")
    print(f"  First order: {first_order_date}")
    print(f"  Total orders: {order_count}")

# 4. HYPOTHESIS TESTING
print(f"\n4. HYPOTHESIS TESTING")
print("-" * 80)

print(f"\nHypothesis A: These are VALID active users, just missing signup_date")
print(f"  Evidence:")
print(f"    ✅ They have orders in clean_sales")
print(f"    ✅ They're in active groups (Regular, Frequent, etc)")
print(f"    ✅ They have wishlist items")
print(f"  Solution: Fill signup_date with first_order_date")

print(f"\nHypothesis B: These are DATA QUALITY ISSUES")
print(f"  Evidence:")
print(f"    ❓ Why would active users have no signup_date?")
print(f"    ❓ Are these duplicates of other buyer_ids?")
print(f"    ❓ Did they get imported wrong?")

# Check for duplicates
print(f"\n5. CHECKING FOR DUPLICATE buyer_ids")
print("-" * 80)

suspicious_ids = suspicious['buyer_id'].unique()
for buyer_id in suspicious_ids[:5]:
    count_in_buyers = len(buyers[buyers['buyer_id'] == buyer_id])
    count_in_sales = len(clean_sales[clean_sales['buyer_id'] == buyer_id])
    
    if count_in_buyers > 1:
        print(f"{buyer_id}: DUPLICATE in buyers ({count_in_buyers} rows)")
    else:
        print(f"{buyer_id}: Unique in buyers, {count_in_sales} orders in sales")

# 6. CHECK SIGNUP vs FIRST ORDER DATES
print(f"\n6. IF WE FILL WITH first_order_date")
print("-" * 80)

# Simulate filling
for buyer_id in suspicious['buyer_id'].unique()[:5]:
    buyer_info = suspicious[suspicious['buyer_id'] == buyer_id].iloc[0]
    first_order_date = clean_sales[clean_sales['buyer_id'] == buyer_id]['order_datetime'].min()
    
    print(f"\n{buyer_id}:")
    print(f"  Current signup_date: NULL")
    print(f"  Would fill with: {first_order_date}")
    print(f"  Makes sense? ✅ YES (signup = first order)")

# 7. RECOMMENDATION
print(f"\n" + "=" * 80)
print("RECOMMENDATION")
print("=" * 80)

print(f"""
For these 101 suspicious buyers:

✅ OPTION 1: FILL with first_order_date
  • These are REAL active users
  • They have orders, so first_order_date is legit
  • Filling makes logical sense
  • Safe to proceed

❌ OPTION 2: REMOVE
  • Would lose 101 active buyers
  • Not recommended unless proved fake

⚠️ OPTION 3: INVESTIGATE FURTHER
  • Check if duplicates exist
  • Verify data source
  • Then decide

DECISION: Fill with first_order_date + Keep
Reason: Real users, valid orders, safe to infer signup_date
""")

print("=" * 80)


INVESTIGATING: 101 SUSPICIOUS BUYERS

1. OVERVIEW
--------------------------------------------------------------------------------
Total suspicious: 1058
Breakdown by group:
customer_group
Regular       445
Occasional    345
Frequent      268
Name: count, dtype: int64

2. THEIR ORDERS IN clean_sales
--------------------------------------------------------------------------------
Orders from suspicious buyers: 11,634
First order date range: 2024-01-01 14:24:00 to 2025-12-30 21:27:00

3. FIRST ORDER FOR EACH SUSPICIOUS BUYER
--------------------------------------------------------------------------------

B29843:
  Customer group: Occasional
  Wishlist size: 8
  First order: 2024-03-30 23:11:00
  Total orders: 6

B11569:
  Customer group: Frequent
  Wishlist size: 6
  First order: 2024-01-09 14:05:00
  Total orders: 28

B32678:
  Customer group: Occasional
  Wishlist size: 15
  First order: 2024-02-28 21:50:00
  Total orders: 6

B15855:
  Customer group: Frequent
  Wishlist size: 19
  Fi

In [ ]:
# SIMPLE FIX: Fill NULL signup_date with first_order_date

print("=" * 80)
print("FILLING NULL signup_date WITH first_order_date")
print("=" * 80)

# 1. GET FIRST ORDER FOR EACH BUYER
print("\n1. Extract first order dates")
print("-" * 80)

first_order = clean_sales.groupby('buyer_id')['order_datetime'].min().reset_index()
first_order.columns = ['buyer_id', 'first_order_date']
print(f"✅ Extracted first orders for {len(first_order):,} buyers")

# 2. MERGE WITH BUYERS
print("\n2. Merge with buyer data")
print("-" * 80)

buyers = buyers.merge(first_order, on='buyer_id', how='left')
print(f"✅ Merged")

# 3. CHECK BEFORE
print("\n3. Before fix")
print("-" * 80)

null_before = buyers['signup_date'].isna().sum()
print(f"NULL signup_date: {null_before:,}")

# 4. FILL NULL signup_date WITH first_order_date
print("\n4. Applying fix")
print("-" * 80)

# Find rows where signup_date is NULL but first_order_date exists
mask = (buyers['signup_date'].isna()) & (buyers['first_order_date'].notna())
filled_count = mask.sum()

# Fill them
buyers.loc[mask, 'signup_date'] = buyers.loc[mask, 'first_order_date']

print(f"✅ Filled {filled_count:,} rows")

# 5. CHECK AFTER
print("\n5. After fix")
print("-" * 80)

null_after = buyers['signup_date'].isna().sum()
print(f"NULL signup_date: {null_after:,}")
print(f"Filled: {filled_count:,}")

# 6. VERIFY EXAMPLES
print("\n6. Examples (before → after)")
print("-" * 80)

# Show some examples
for buyer_id in ['B11569', 'B15855', 'B15557']:
    if buyer_id in buyers['buyer_id'].values:
        buyer = buyers[buyers['buyer_id'] == buyer_id].iloc[0]
        print(f"\n{buyer_id}:")
        print(f"  signup_date (now): {buyer['signup_date']}")
        print(f"  first_order_date: {buyer['first_order_date']}")

# 7. CLEANUP (remove temporary column)
print("\n7. Cleanup")
print("-" * 80)

buyers = buyers.drop('first_order_date', axis=1).copy()
print(f"✅ Dropped first_order_date column (no longer needed)")

# 8. FINAL CHECK
print("\n8. Final verification")
print("-" * 80)

print(f"NULL signup_date: {buyers['signup_date'].isna().sum()}")
print(f"Date range: {buyers['signup_date'].min().date()} to {buyers['signup_date'].max().date()}")

print(f"\n✅ DONE - signup_date FIXED!")
print("=" * 80)


FILLING NULL signup_date WITH first_order_date

1. Extract first order dates
--------------------------------------------------------------------------------
✅ Extracted first orders for 25,000 buyers

2. Merge with buyer data
--------------------------------------------------------------------------------
✅ Merged

3. Before fix
--------------------------------------------------------------------------------
NULL signup_date: 2,431

4. Applying fix
--------------------------------------------------------------------------------
✅ Filled 1,121 rows

5. After fix
--------------------------------------------------------------------------------
NULL signup_date: 1,310
Filled: 1,121

6. Examples (before → after)
--------------------------------------------------------------------------------

B11569:
  signup_date (now): 2024-01-09 14:05:00
  first_order_date: 2024-01-09 14:05:00

B15855:
  signup_date (now): 2024-01-20 05:59:00
  first_order_date: 2024-01-20 05:59:00

B15557:
  signup_dat

In [ ]:
# JUST CHECK: What are the remaining 1,310 NULL signup_date?

print("=" * 80)
print("CHECKING REMAINING NULL signup_date")
print("=" * 80)

remaining_null = buyers[buyers['signup_date'].isna()]
print(f"\nRemaining NULL: {len(remaining_null)}")

# Check if they're in sales
remaining_ids = set(remaining_null['buyer_id'].unique())
sales_ids = set(clean_sales['buyer_id'].unique())

has_orders = remaining_ids & sales_ids
no_orders = remaining_ids - sales_ids

print(f"\nBreakdown:")
print(f"  Have orders in clean_sales: {len(has_orders)}")
print(f"  NO orders: {len(no_orders)}")

if len(has_orders) > 0:
    print(f"\n❌ {len(has_orders)} have orders but NULL signup_date!")
    
if len(no_orders) > 0:
    print(f"\n✅ {len(no_orders)} have NO orders (dormant)")
    dormant = remaining_null[remaining_null['buyer_id'].isin(no_orders)]
    print(f"   Groups: {dormant['customer_group'].value_counts().to_dict()}")

print("\n" + "=" * 80)


CHECKING REMAINING NULL signup_date

Remaining NULL: 1310

Breakdown:
  Have orders in clean_sales: 0
  NO orders: 1140

✅ 1140 have NO orders (dormant)
   Groups: {'Dormant': 1209, 'Regular': 37, 'Occasional': 32, 'Frequent': 24, 'VIP': 8}



In [ ]:
suspicious_ids = buyers[
    buyers['signup_date'].isna() & 
    ~buyers['buyer_id'].isin(clean_sales['buyer_id']) &
    buyers['customer_group'].isin(['Regular', 'Occasional', 'Frequent', 'VIP'])
]['buyer_id']

# Are these buyer_ids duplicated in the buyer table?
print(buyers[buyers['buyer_id'].isin(suspicious_ids)][
    ['buyer_id', 'customer_group', 'signup_date', 'wishlist_size']
].sort_values('buyer_id').head(20))

       buyer_id customer_group signup_date  wishlist_size
29264   B10133             VIP         NaT              4
9567    B10781             VIP         NaT             14
3594    B11136             VIP         NaT              8
45239   B11550        Frequent         NaT              9
6329    B12263        Frequent         NaT             15
292     B12533        Frequent         NaT              7
29006   B13353        Frequent         NaT              4
44506   B14316        Frequent         NaT             15
38234   B14594        Frequent         NaT             11
13876   B15060        Frequent         NaT             24
8767    B15080        Frequent         NaT              2
10428   B16247        Frequent         NaT              8
22790   B17112        Frequent         NaT              9
43235   B17758         Regular         NaT              9
1290    B18314         Regular         NaT             10
6347    B19015         Regular         NaT              9
41079   B20094

In [ ]:
# How many are we talking about exactly?
problem_buyers = buyers[
    buyers['signup_date'].isna() & 
    ~buyers['buyer_id'].isin(clean_sales['buyer_id']) &
    buyers['customer_group'].isin(['Regular', 'Occasional', 'Frequent', 'VIP'])
]
print(f"Total: {len(problem_buyers)}")
print(problem_buyers['customer_group'].value_counts())

Total: 101
customer_group
Regular       37
Occasional    32
Frequent      24
VIP            8
Name: count, dtype: int64


In [ ]:
# Reclassify to Dormant
mask = (
    buyers['signup_date'].isna() & 
    ~buyers['buyer_id'].isin(clean_sales['buyer_id']) &
    buyers['customer_group'].isin(['Regular', 'Occasional', 'Frequent', 'VIP'])
)

buyers.loc[mask, 'customer_group'] = 'Dormant'

# Verify
print(f"Reclassified: {mask.sum()} buyers")
print(buyers['customer_group'].value_counts())

Reclassified: 101 buyers
customer_group
Dormant       26371
Regular       10451
Occasional     7820
Frequent       6287
VIP            1571
Name: count, dtype: int64


In [ ]:
# CHECK: Current NULL signup_date after fixes

print("=" * 80)
print("CURRENT NULL signup_date STATUS")
print("=" * 80)

# 1. COUNT NULLS
print("\n1. NULL signup_date COUNT")
print("-" * 80)

null_count = buyers['signup_date'].isna().sum()
total_rows = len(buyers)
null_pct = null_count / total_rows * 100

print(f"Total rows: {total_rows:,}")
print(f"NULL signup_date: {null_count:,}")
print(f"Valid signup_date: {total_rows - null_count:,}")
print(f"NULL %: {null_pct:.2f}%")

# 2. IF ANY NULLS, WHAT ARE THEY?
print("\n2. CHARACTERISTICS OF REMAINING NULL")
print("-" * 80)

if null_count > 0:
    null_rows = buyers[buyers['signup_date'].isna()]
    
    print(f"Customer groups:")
    print(null_rows['customer_group'].value_counts())
    
    print(f"\nIn clean_sales (have orders)?")
    in_sales = null_rows['buyer_id'].isin(clean_sales['buyer_id']).sum()
    print(f"  Yes: {in_sales}")
    print(f"  No: {null_count - in_sales}")
    
    if in_sales > 0:
        print(f"\n⚠️ WARNING: {in_sales} NULL signup_date still have orders!")

# 3. SUMMARY OF ACTIONS
print("\n3. SUMMARY OF ACTIONS TAKEN")
print("-" * 80)

print(f"""
Actions:
  ✅ Filled 1,121 from first_order_date
  ✅ Reclassified 101 to Dormant
  
Current state:
  • Total rows: {total_rows:,}
  • NULL signup_date: {null_count:,} ({null_pct:.2f}%)
  • Valid signup_date: {total_rows - null_count:,}
""")

# 4. STATUS
print("\n4. STATUS")
print("-" * 80)

if null_count == 0:
    print("✅ COMPLETE - All signup_date values filled!")
elif null_pct < 1:
    print(f"✅ GOOD - Only {null_pct:.2f}% NULL (dormant users acceptable)")
else:
    print(f"⚠️ REVIEW - {null_count:,} NULL remaining")

print("=" * 80)


CURRENT NULL signup_date STATUS

1. NULL signup_date COUNT
--------------------------------------------------------------------------------
Total rows: 52,500
NULL signup_date: 1,310
Valid signup_date: 51,190
NULL %: 2.50%

2. CHARACTERISTICS OF REMAINING NULL
--------------------------------------------------------------------------------
Customer groups:
customer_group
Dormant    1310
Name: count, dtype: int64

In clean_sales (have orders)?
  Yes: 0
  No: 1310

3. SUMMARY OF ACTIONS TAKEN
--------------------------------------------------------------------------------

Actions:
  ✅ Filled 1,121 from first_order_date
  ✅ Reclassified 101 to Dormant
  
Current state:
  • Total rows: 52,500
  • NULL signup_date: 1,310 (2.50%)
  • Valid signup_date: 51,190


4. STATUS
--------------------------------------------------------------------------------
⚠️ REVIEW - 1,310 NULL remaining


In [ ]:
# Show rows where buyer_id is duplicated
dup_buyer_rows = buyers[buyers.duplicated(subset=["buyer_id"], keep=False)].copy()

print("Duplicate buyer_id rows:", len(dup_buyer_rows))
print("Unique duplicate buyer_ids:", dup_buyer_rows["buyer_id"].nunique())

# View duplicate rows
dup_buyer_rows.sort_values("buyer_id").head(100)



Duplicate buyer_id rows: 7366
Unique duplicate buyer_ids: 2366


,buyer_id,customer_group,signup_date,customer_segment,preferred_subcategory,subcategory_pool,preferred_channel,preferred_payment,region,state,timezone,is_referred,wishlist_size,in_sales
12844,B10145,VIP,2023-12-07,Fitness Buff,Water Bottles,"Smart Bands, Supplements",App,Digital Wallet,West,NV,America/Los_Angeles,0,11,1
45912,B10145,VIP,2023-12-07,Fitness Buff,Water Bottles,"Smart Bands, Supplements",App,Digital Wallet,West,NV,America/Los_Angeles,0,11,1
2977,B10439,VIP,2023-09-20,Beauty Lover,Makeup Brushes,"Makeup Brushes, Nail Art",Website,Digital Wallet,West,NV,America/Los_Angeles,0,6,1
27259,B10439,VIP,2023-09-20,Beauty Lover,Makeup Brushes,"Makeup Brushes, Nail Art",Website,Digital Wallet,West,NV,America/Los_Angeles,0,6,1
32597,B10567,VIP,2023-12-07,Tech Enthusiast,Earphones,"Smartphones, Power Banks",App,Credit Card,East,NJ,America/New_York,0,10,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34351,B26964,Regular,2022-02-22,Beauty Lover,Makeup Brushes,"Lipsticks, Perfumes",Website,Digital Wallet,West,OR,America/Los_Angeles,0,3,0
29562,B26979,Regular,2021-07-05,Bargain Hunter,Phone Cases,"Stationery, Socks",App,Credit Card,West,OR,America/Los_Angeles,0,10,0
4085,B26979,Regular,2021-07-05,Bargain Hunter,Phone Cases,"Stationery, Socks",App,Credit Card,West,OR,America/Los_Angeles,0,10,0
39203,B27698,Occasional,2020-09-06,Tech Enthusiast,Power Banks,"Smart Watches, Power Banks",App,Digital Wallet,East,NY,America/New_York,0,7,1


In [ ]:
# Only identify duplicate buyer_id rows (no completeness scoring, no dedup)

dup_buyer_rows = buyers[buyers.duplicated(subset=["buyer_id"], keep=False)].copy()

print("Duplicate buyer_id rows:", len(dup_buyer_rows))
print("Unique duplicate buyer_ids:", dup_buyer_rows["buyer_id"].nunique())

# View duplicate rows
dup_buyer_rows.sort_values("buyer_id").head(100)


Duplicate buyer_id rows: 7366
Unique duplicate buyer_ids: 2366


,buyer_id,customer_group,signup_date,customer_segment,preferred_subcategory,subcategory_pool,preferred_channel,preferred_payment,region,state,timezone,is_referred,wishlist_size,in_sales
12844,B10145,VIP,2023-12-07,Fitness Buff,Water Bottles,"Smart Bands, Supplements",App,Digital Wallet,West,NV,America/Los_Angeles,0,11,1
45912,B10145,VIP,2023-12-07,Fitness Buff,Water Bottles,"Smart Bands, Supplements",App,Digital Wallet,West,NV,America/Los_Angeles,0,11,1
2977,B10439,VIP,2023-09-20,Beauty Lover,Makeup Brushes,"Makeup Brushes, Nail Art",Website,Digital Wallet,West,NV,America/Los_Angeles,0,6,1
27259,B10439,VIP,2023-09-20,Beauty Lover,Makeup Brushes,"Makeup Brushes, Nail Art",Website,Digital Wallet,West,NV,America/Los_Angeles,0,6,1
32597,B10567,VIP,2023-12-07,Tech Enthusiast,Earphones,"Smartphones, Power Banks",App,Credit Card,East,NJ,America/New_York,0,10,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34351,B26964,Regular,2022-02-22,Beauty Lover,Makeup Brushes,"Lipsticks, Perfumes",Website,Digital Wallet,West,OR,America/Los_Angeles,0,3,0
29562,B26979,Regular,2021-07-05,Bargain Hunter,Phone Cases,"Stationery, Socks",App,Credit Card,West,OR,America/Los_Angeles,0,10,0
4085,B26979,Regular,2021-07-05,Bargain Hunter,Phone Cases,"Stationery, Socks",App,Credit Card,West,OR,America/Los_Angeles,0,10,0
39203,B27698,Occasional,2020-09-06,Tech Enthusiast,Power Banks,"Smart Watches, Power Banks",App,Digital Wallet,East,NY,America/New_York,0,7,1


In [ ]:
# Among rows with duplicate buyer_id, count exact duplicates (all columns identical)

dup_buyer_rows = buyers[buyers.duplicated(subset=["buyer_id"], keep=False)].copy()

exact_dup_mask = dup_buyer_rows.duplicated(keep=False)
exact_dup_rows = dup_buyer_rows[exact_dup_mask]

print("Rows with duplicate buyer_id:", len(dup_buyer_rows))
print("Exact duplicate rows (all columns same):", len(exact_dup_rows))
print("Unique exact-duplicate patterns:", exact_dup_rows.drop_duplicates().shape[0])


Rows with duplicate buyer_id: 7366
Exact duplicate rows (all columns same): 5000
Unique exact-duplicate patterns: 2500


In [ ]:
buyers = buyers.drop_duplicates(keep="first").copy()


In [ ]:
# Check how many of non-exact rows have null buyer_id
print("Non-exact rows:", len(non_exact))
print("Null buyer_id in non-exact:", non_exact["buyer_id"].isna().sum())
print("Non-null buyer_id in non-exact:", non_exact["buyer_id"].notna().sum())

# Include NaN as a group
cols_to_check = [c for c in buyers.columns if c != "buyer_id"]

diff_summary = []
for bid, g in non_exact.groupby("buyer_id", dropna=False):
    for col in cols_to_check:
        if g[col].nunique(dropna=False) > 1:
            diff_summary.append((bid, col, g[col].nunique(dropna=False)))

diff_summary_df = pd.DataFrame(
    diff_summary, columns=["buyer_id", "column_that_differs", "unique_values"]
)

print("\nMost frequently differing columns:")
print(diff_summary_df["column_that_differs"].value_counts() if not diff_summary_df.empty else "No differing columns found")


Non-exact rows: 2500
Null buyer_id in non-exact: 2500
Non-null buyer_id in non-exact: 0

Most frequently differing columns:
column_that_differs
customer_group           1
signup_date              1
customer_segment         1
preferred_subcategory    1
subcategory_pool         1
preferred_channel        1
preferred_payment        1
region                   1
state                    1
timezone                 1
is_referred              1
wishlist_size            1
Name: count, dtype: int64


In [ ]:
buyers = buyers[buyers["buyer_id"].notna()].copy()


In [ ]:
# Check missing values across all columns in deduplicated buyers table
# (run after buyers_dedup is created)

missing_summary = buyers.isna().sum().sort_values(ascending=False)
missing_summary = missing_summary[missing_summary > 0]

print("Columns with missing values:")
print(missing_summary)

# Specifically check missing buyer_id
print("\nMissing buyer_id:", buyers["buyer_id"].isna().sum())


Columns with missing values:
signup_date    1139
dtype: int64

Missing buyer_id: 0


In [ ]:
# Check how many rows with missing signup_date are Dormant buyers
# (assumes customer_group has values like "Dormant")

missing_signup = buyers[buyers["signup_date"].isna()].copy()

dormant_count = (missing_signup["customer_group"].astype(str).str.strip().str.lower() == "dormant").sum()
total_missing_signup = len(missing_signup)

print("Rows with missing signup_date:", total_missing_signup)
print("Dormant among missing signup_date:", dormant_count)
print("Dormant % among missing signup_date:", round(dormant_count / total_missing_signup * 100, 2) if total_missing_signup else 0)


Rows with missing signup_date: 1139
Dormant among missing signup_date: 1139
Dormant % among missing signup_date: 100.0


In [ ]:
# Investigate only (no fixes) for invalid categorical encoding in is_referred

# 1) Value distribution
print("is_referred value counts:")
print(buyers["is_referred"].value_counts(dropna=False).sort_index())

# 2) Invalid rows (anything not 0 or 1, excluding nulls)
invalid_mask = buyers["is_referred"].notna() & ~buyers["is_referred"].isin([0, 1])
invalid_rows = buyers[invalid_mask].copy()

print("\nInvalid is_referred rows:", len(invalid_rows))

# 3) See full invalid records
display(invalid_rows)

# 4) Quick profile of invalid rows
print("\nInvalid rows by customer_group:")
print(invalid_rows["customer_group"].value_counts(dropna=False))

print("\nInvalid rows by region:")
print(invalid_rows["region"].value_counts(dropna=False))

print("\nInvalid rows with null buyer_id:")
print(invalid_rows["buyer_id"].isna().sum())


is_referred value counts:
is_referred
0    40043
1     6984
2      473
Name: count, dtype: int64

Invalid is_referred rows: 473


,buyer_id,customer_group,signup_date,customer_segment,preferred_subcategory,subcategory_pool,preferred_channel,preferred_payment,region,state,timezone,is_referred,wishlist_size,in_sales
339,B34087,Occasional,2023-06-18,Bargain Hunter,Socks,"Stationery, Small Accessories",App,Digital Wallet,Central,IL,America/Chicago,2,14,1
611,B48178,Dormant,2022-02-27,Beauty Lover,Nail Art,"Nail Art, Lipsticks",Mobile,Digital Wallet,East,GA,America/New_York,2,9,0
778,B57527,Dormant,2023-12-30,Fitness Buff,Yoga Mats,"Supplements, Yoga Mats",Website,Digital Wallet,West,OR,America/Los_Angeles,2,12,0
788,B45053,Dormant,2023-02-02,Tech Enthusiast,Bluetooth Speakers,"Smartphones, Power Banks",Mobile,Digital Wallet,Central,WI,America/Chicago,2,11,0
809,B32024,Occasional,2023-10-22,Home Organizer,Trash Bins,"Bathroom Organizer, Cleaning Tools",Website,Other,West,WA,America/Los_Angeles,2,11,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51505,B49591,Dormant,2020-05-28,Fashion Shopper,Sunglasses,"Women’s Tops, Sunglasses",Website,Other,West,NV,America/Los_Angeles,2,29,0
51572,B25247,Regular,2022-11-14,Fitness Buff,Yoga Mats,"Water Bottles, Supplements",Mobile,Digital Wallet,East,PA,America/New_York,2,10,1
52015,B44429,Dormant,2018-05-02,Bargain Hunter,Socks,"Phone Cases, Stationery",App,Digital Wallet,East,PA,America/New_York,2,13,0
52147,B56451,Dormant,2022-05-06,Home Organizer,Trash Bins,"Bathroom Organizer, Kitchen Storage",App,Digital Wallet,East,NJ,America/New_York,2,14,0



Invalid rows by customer_group:
customer_group
Dormant       259
Regular        86
Occasional     69
Frequent       47
VIP            12
Name: count, dtype: int64

Invalid rows by region:
region
West       182
East       163
Central     91
Other       37
Name: count, dtype: int64

Invalid rows with null buyer_id:
0


In [ ]:
# Common patterns in rows where is_referred == 2

bad2 = buyers[buyers["is_referred"] == 2].copy()
print("Rows with is_referred=2:", len(bad2))

# 1) Show the rows
display(bad2)

# 2) For each column, check if all values are the same (common value)
common_summary = []
for col in bad2.columns:
    nunq = bad2[col].nunique(dropna=False)
    if nunq == 1:
        common_summary.append((col, bad2[col].iloc[0], "ALL SAME"))
    else:
        common_summary.append((col, f"{nunq} unique", "VARIES"))

common_df = pd.DataFrame(common_summary, columns=["column", "value_or_unique_count", "status"])
display(common_df)

# 3) Value counts per column (to see dominant/common values)
for col in bad2.columns:
    print(f"\n--- {col} ---")
    print(bad2[col].value_counts(dropna=False))


Rows with is_referred=2: 473


,buyer_id,customer_group,signup_date,customer_segment,preferred_subcategory,subcategory_pool,preferred_channel,preferred_payment,region,state,timezone,is_referred,wishlist_size,in_sales
339,B34087,Occasional,2023-06-18,Bargain Hunter,Socks,"Stationery, Small Accessories",App,Digital Wallet,Central,IL,America/Chicago,2,14,1
611,B48178,Dormant,2022-02-27,Beauty Lover,Nail Art,"Nail Art, Lipsticks",Mobile,Digital Wallet,East,GA,America/New_York,2,9,0
778,B57527,Dormant,2023-12-30,Fitness Buff,Yoga Mats,"Supplements, Yoga Mats",Website,Digital Wallet,West,OR,America/Los_Angeles,2,12,0
788,B45053,Dormant,2023-02-02,Tech Enthusiast,Bluetooth Speakers,"Smartphones, Power Banks",Mobile,Digital Wallet,Central,WI,America/Chicago,2,11,0
809,B32024,Occasional,2023-10-22,Home Organizer,Trash Bins,"Bathroom Organizer, Cleaning Tools",Website,Other,West,WA,America/Los_Angeles,2,11,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51505,B49591,Dormant,2020-05-28,Fashion Shopper,Sunglasses,"Women’s Tops, Sunglasses",Website,Other,West,NV,America/Los_Angeles,2,29,0
51572,B25247,Regular,2022-11-14,Fitness Buff,Yoga Mats,"Water Bottles, Supplements",Mobile,Digital Wallet,East,PA,America/New_York,2,10,1
52015,B44429,Dormant,2018-05-02,Bargain Hunter,Socks,"Phone Cases, Stationery",App,Digital Wallet,East,PA,America/New_York,2,13,0
52147,B56451,Dormant,2022-05-06,Home Organizer,Trash Bins,"Bathroom Organizer, Kitchen Storage",App,Digital Wallet,East,NJ,America/New_York,2,14,0


,column,value_or_unique_count,status
0,buyer_id,473 unique,VARIES
1,customer_group,5 unique,VARIES
2,signup_date,363 unique,VARIES
3,customer_segment,8 unique,VARIES
4,preferred_subcategory,43 unique,VARIES
5,subcategory_pool,174 unique,VARIES
6,preferred_channel,3 unique,VARIES
7,preferred_payment,4 unique,VARIES
8,region,4 unique,VARIES
9,state,21 unique,VARIES



--- buyer_id ---
buyer_id
B34087    1
B38033    1
B46613    1
B42153    1
B17042    1
         ..
B10229    1
B26450    1
B13122    1
B53614    1
B12158    1
Name: count, Length: 473, dtype: int64

--- customer_group ---
customer_group
Dormant       259
Regular        86
Occasional     69
Frequent       47
VIP            12
Name: count, dtype: int64

--- signup_date ---
signup_date
NaT           13
2023-12-30     9
2023-04-29     5
2023-12-26     5
2023-11-06     4
              ..
2018-09-04     1
2023-09-12     1
2019-10-03     1
2023-03-24     1
2023-10-28     1
Name: count, Length: 363, dtype: int64

--- customer_segment ---
customer_segment
Beauty Lover       77
Bargain Hunter     59
Fashion Shopper    59
Young Parent       59
Fitness Buff       57
Cozy Homemaker     57
Tech Enthusiast    55
Home Organizer     50
Name: count, dtype: int64

--- preferred_subcategory ---
preferred_subcategory
Makeup Brushes           17
Home Fragrance           16
Sunglasses               16
Socks 

In [ ]:
# 1) Flag invalid encoding
buyers["is_referred_invalid"] = ~buyers["is_referred"].isin([0, 1])

# 2) Set invalid values to NaN (safer than forcing 0/1)
buyers.loc[buyers["is_referred_invalid"], "is_referred"] = pd.NA

# 3) Keep column as nullable integer
buyers["is_referred"] = buyers["is_referred"].astype("Int64")

# 4) Check
print("Invalid flagged rows:", buyers["is_referred_invalid"].sum())
print(buyers["is_referred"].value_counts(dropna=False))


Invalid flagged rows: 473
is_referred
0       40043
1        6984
<NA>      473
Name: count, dtype: Int64


In [ ]:
# FINAL BUYERS EDA QA - verify documented issues across all columns
# Assumes your final cleaned dataframe is `buyers`

import pandas as pd
import numpy as np

df = buyers.copy()

# ---------- normalize dtypes for checks ----------
if "signup_date" in df.columns:
    df["signup_date"] = pd.to_datetime(df["signup_date"], errors="coerce")

# helper: safe missing count
def miss(col):
    return int(df[col].isna().sum()) if col in df.columns else np.nan

# helper: safe duplicate-row count by key
def dup_rows(key_cols):
    if all(c in df.columns for c in key_cols):
        return int(df.duplicated(subset=key_cols, keep=False).sum())
    return np.nan

# ---------- issue checks ----------
# 1) Missing primary identifier
issue1 = miss("buyer_id")

# 2) Duplicate buyer IDs (one-row-per-buyer expectation)
issue2 = dup_rows(["buyer_id"])

# 3) Missing region
issue3 = miss("region")

# 4) Missing timezone
issue4 = miss("timezone")

# 5) Missing signup_date
issue5 = miss("signup_date")

# 6) Invalid categorical encoding is_referred not in {0,1}
if "is_referred" in df.columns:
    invalid_is_referred = (~df["is_referred"].isin([0, 1])) & (df["is_referred"].notna())
    issue6 = int(invalid_is_referred.sum())
else:
    issue6 = np.nan

# 7) Redundant feature check: is_active_buyer duplicates customer_group(Dormant vs not Dormant)
if all(c in df.columns for c in ["is_active_buyer", "customer_group"]):
    derived_active = (~df["customer_group"].astype(str).str.strip().str.lower().eq("dormant")).astype("Int64")
    mismatch_mask = (
        df["is_active_buyer"].notna()
        & derived_active.notna()
        & (df["is_active_buyer"].astype("Int64") != derived_active)
    )
    issue7 = int(mismatch_mask.sum())   # 0 means perfect duplication/overlap
else:
    issue7 = np.nan

# ---------- summary table ----------
qa = pd.DataFrame([
    {"issue_no": 1, "issue": "Missing Primary Identifier (buyer_id)", "remaining_issue_rows": issue1, "status": "PASS" if issue1 == 0 else "FAIL"},
    {"issue_no": 2, "issue": "Duplicate buyer_id", "remaining_issue_rows": issue2, "status": "PASS" if issue2 == 0 else "FAIL"},
    {"issue_no": 3, "issue": "Missing region", "remaining_issue_rows": issue3, "status": "PASS" if issue3 == 0 else "FAIL"},
    {"issue_no": 4, "issue": "Missing timezone", "remaining_issue_rows": issue4, "status": "PASS" if issue4 == 0 else "FAIL"},
    {"issue_no": 5, "issue": "Missing signup_date", "remaining_issue_rows": issue5, "status": "PASS" if issue5 == 0 else "FAIL"},
    {"issue_no": 6, "issue": "Invalid is_referred encoding (!=0/1)", "remaining_issue_rows": issue6, "status": "PASS" if issue6 == 0 else "FAIL"},
    {
        "issue_no": 7,
        "issue": "Redundancy: is_active_buyer duplicates customer_group",
        "remaining_issue_rows": issue7,
        "status": "PASS" if issue7 > 0 else "INFO: still perfectly redundant"
    },
]).sort_values("issue_no")

print("FINAL BUYERS DATA QUALITY CHECK")
print("-" * 80)
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
display(qa)

# ---------- optional: detail blocks ----------
print("\nMissing values by column (non-zero only):")
display(df.isna().sum()[df.isna().sum() > 0].sort_values(ascending=False).to_frame("missing_rows"))

if "is_referred" in df.columns:
    print("\nis_referred distribution:")
    print(df["is_referred"].value_counts(dropna=False))

if all(c in df.columns for c in ["customer_group", "is_active_buyer"]):
    print("\ncustomer_group vs is_active_buyer crosstab:")
    display(pd.crosstab(df["customer_group"], df["is_active_buyer"], dropna=False))


FINAL BUYERS DATA QUALITY CHECK
--------------------------------------------------------------------------------
Rows: 47,500
Columns: 15


,issue_no,issue,remaining_issue_rows,status
0,1,Missing Primary Identifier (buyer_id),0.0,PASS
1,2,Duplicate buyer_id,0.0,PASS
2,3,Missing region,0.0,PASS
3,4,Missing timezone,0.0,PASS
4,5,Missing signup_date,1139.0,FAIL
5,6,Invalid is_referred encoding (!=0/1),0.0,PASS
6,7,Redundancy: is_active_buyer duplicates custome...,NaN,INFO: still perfectly redundant



Missing values by column (non-zero only):


,missing_rows
signup_date,1139
is_referred,473



is_referred distribution:
is_referred
0       40043
1        6984
<NA>      473
Name: count, dtype: Int64


In [39]:
from pathlib import Path

out_dir = Path("processed_data")
out_dir.mkdir(parents=True, exist_ok=True)

buyers.to_csv(out_dir / "buyers_clean.csv", index=False)
buyers.to_parquet(out_dir / "buyers_clean.parquet", index=False)

# quick verify from saved file
import pandas as pd
b = pd.read_csv(out_dir / "buyers_clean.csv")
print("missing timezone:", b["timezone"].isna().sum())      # should be 0
print("missing signup_date:", b["signup_date"].isna().sum()) # likely 1139


missing timezone: 0
missing signup_date: 2431


In [40]:
from pathlib import Path
import pandas as pd

out_dir = Path("processed_data")

# 1) Check current in-memory dataframe
print("IN-MEMORY buyers")
print("rows:", len(buyers))
print("signup nulls (raw):", buyers["signup_date"].isna().sum())

tmp = buyers.copy()
tmp["signup_date"] = pd.to_datetime(tmp["signup_date"], format="%m/%d/%y %H:%M", errors="coerce")
print("signup nulls (parsed):", tmp["signup_date"].isna().sum())

# 2) Save exactly this verified version
tmp.to_csv(out_dir / "buyers_clean.csv", index=False)
tmp.to_parquet(out_dir / "buyers_clean.parquet", index=False)

# 3) Reload and recheck
b = pd.read_csv(out_dir / "buyers_clean.csv")
b["signup_date"] = pd.to_datetime(b["signup_date"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
print("\nRELOADED buyers_clean.csv")
print("rows:", len(b))
print("timezone nulls:", b["timezone"].isna().sum())
print("signup nulls:", b["signup_date"].isna().sum())


IN-MEMORY buyers
rows: 52500
signup nulls (raw): 2431
signup nulls (parsed): 2431

RELOADED buyers_clean.csv
rows: 52500
timezone nulls: 0
signup nulls: 52500


In [ ]:
# Cell 49: Whitespace Fix for buyer_id (buyers + clean_sales) and verification
from pathlib import Path
import pandas as pd

out_dir = Path('processed_data')

# Normalize in-memory buyers
buyers['buyer_id'] = buyers['buyer_id'].astype('string').str.strip()

# Normalize clean_sales buyer_id in memory (load if needed)
if 'clean_sales' not in globals():
    if (out_dir / 'sales_clean.parquet').exists():
        clean_sales = pd.read_parquet(out_dir / 'sales_clean.parquet')
    elif (out_dir / 'sales_clean.csv').exists():
        clean_sales = pd.read_csv(out_dir / 'sales_clean.csv')
    elif (out_dir / 'clean_sales.csv').exists():  # backward compatibility
        clean_sales = pd.read_csv(out_dir / 'clean_sales.csv')
    else:
        raise FileNotFoundError('Missing sales_clean/clean_sales in processed_data')

clean_sales['buyer_id'] = clean_sales['buyer_id'].astype('string').str.strip()

# Verify whitespace removed
buyers_ws = buyers['buyer_id'].str.len().ne(buyers['buyer_id'].str.strip().str.len()).sum()
sales_ws = clean_sales['buyer_id'].str.len().ne(clean_sales['buyer_id'].str.strip().str.len()).sum()
print('buyers leading/trailing spaces:', int(buyers_ws))
print('sales leading/trailing spaces:', int(sales_ws))

# Optional: persist normalized buyers_clean immediately
buyers.to_csv(out_dir / 'buyers_clean.csv', index=False)
buyers.to_parquet(out_dir / 'buyers_clean.parquet', index=False)
print('Saved buyers_clean with trimmed buyer_id')
